# D4_noisy_simulation — 雑音付きモンテカルロ（Kobayashi 方式の出力誤り・表面符号の論理誤り率）

**目的**：物理誤り率 $p$ の回路レベル雑音の下で、
1. Kobayashi–Zhu–Hsin（[KZH]）の D4 code switching が出力する論理魔法状態 $T^\dagger|+\rangle_L$ の誤り（不忠実度）$\varepsilon_K(d,p)$ を求める（非 Clifford 回路の厳密シミュレーション）。
2. 比較相手（Litinski の蒸留）の誤り予算に使う回転表面符号の論理誤り率 $p_L(d,p)$ を、**同じ雑音モデル**で較正する。

**出力**：`results/d4_noisy_mc.csv`（ショットの塊ごとの生データ）、`results/surface_code_plog.csv`。比較・費用計算は `noisy_cost_comparison.ipynb` が行う。
`QUICK = True`（既定）は数分の動作確認で、結果は `*_quick.csv` に書かれ、リポジトリの本番結果を上書きしない。本番結果（リポジトリ同梱）は `QUICK = False`、4 コアで約 5 時間（診断用の点を含む）。

**方法の要点**（詳細は `docs/methods.md`）

| 項目 | 内容 |
|---|---|
| 回路 | v4 で検証済みの D4 符号・測定ガジェット（CCZ を含む）・transversal $U=WV$ を、段階 S0（FSC $|+_L\rangle$ 準備）→ S1（D4 安定化子のラウンド、各ラウンドは Z 検査 → X 型 Clifford 安定化子の順）→ U 直前の Z 検査ラウンド → U → switch-out（$Z^g$ 読み出し）→ S4（FSC ラウンド）→ 雑音なしの最終ラウンド、の順に時間ステップ単位で構成 |
| 雑音 | $k$ 量子ビットゲートの後に $k$ 量子ビット脱分極（確率 $p$、CCZ・CS・$T^\dagger$ も $p$）、初期化後・測定前に反転 $p$、アイドル量子ビットに脱分極 $p_{\rm idle}$。シナリオ A：$p_{\rm idle}=p$、シナリオ B：$p_{\rm idle}=0$（Takada らと同じ） |
| 非 Clifford シミュレーション | **Takada–Bartlett–Williamson の Clifford-stabilizer simulation（Pauli proxy 法, arXiv:2609.16929, Algorithm 2）を採用**。適用条件（Definition 2, Sec. VIII）を D4 code switching について確認し（`validation.ipynb` N1）、さらに $L=1,2$ で非 Clifford 回路の厳密な状態ベクトル計算と全測定で一致することを確認した（N3） |
| 復号 | flux（$Z$ 検査）：毎ラウンドの実時間 MWPM と即時 $X$ 補正（JIT 型、本研究の構成）。charge（$X$ 型検査）：プロトコル終了後に stim の誤りモデル＋MWPM（switch-out をまたぐ検出器の基底を 2 通り試し重みの小さい方を採る）、flux の位置から乱された測定の重みを下げる。[Dav] の JIT decoder は参照できなかったため**本研究独自の復号器**であり、最適ではない |
| 論理誤り | 最後に雑音なしのラウンドで復号した後、出力の論理 Clifford 安定化子 $\bar H$ の期待値から不忠実度 $(1-\langle\bar H\rangle)/2$（Takada Sec. VI D） |

In [ ]:
# ==== 環境設定（Google Colab / ローカル共通）====
# このノートブックは単独で実行できる（共通コードは下に埋め込まれており、他のファイルを読み込まない）。
# 出力（results/, figures/）はこのランタイムの作業ディレクトリに保存される。Colab ではランタイム終了で
# 消えるので、残したい場合は Google Drive を使う：
#   from google.colab import drive; drive.mount('/content/drive')
#   os.makedirs('/content/drive/MyDrive/D4_vs_MSD', exist_ok=True); os.chdir('/content/drive/MyDrive/D4_vs_MSD')
import os, sys, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
RESULTS, FIGURES = 'results', 'figures'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
SEED = 20261024
print('cwd =', os.getcwd(), '| Colab =', IN_COLAB)

In [ ]:
# ==== 外部ライブラリ（Colab に無いものだけをインストール）====
import importlib.util
for _mod, _pkg in [('stim', 'stim'), ('pymatching', 'pymatching'), ('mpmath', 'mpmath'), ('scipy', 'scipy'),
                   ('sympy', 'sympy'), ('pandas', 'pandas'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)
import numpy as np, pandas as pd, scipy, stim, pymatching, mpmath
print('numpy', np.__version__, '| scipy', scipy.__version__, '| stim', stim.__version__,
      '| pymatching', pymatching.__version__, '| mpmath', mpmath.__version__)

## 共通ライブラリ（埋め込み・関数定義のみ）
Google Colab ではノートブックごとに別のランタイムで実行されるため、他のファイルを読み込まずに済むよう、共通コードをこのノートブックに埋め込んでいる（各ノートの共通コードは同一のソースから自動生成されており、内容は一致する）。
**このセクションは上から順に実行するだけでよく、編集する必要はない**（Colab では見出しの左の三角で折りたためる）。

### 共通ライブラリ（F2 線形代数・ゲート／ガジェット・スケジューラ・Clifford+T 分解・計数ルール R1–R3）

In [ ]:
"""common_lib: F2 linear algebra, gate/gadget model, scheduler, decompositions, cost model rules."""
import itertools, math
import numpy as np
from collections import Counter, defaultdict

In [ ]:
# ---------------------------------------------------------------- F2 algebra
def f2_rref(M):
    """Row-reduce a uint8 matrix over F2. Returns (R, pivot_cols)."""
    M = (np.array(M, dtype=np.uint8) & 1).copy()
    rows, cols = M.shape
    piv = []
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        nz = np.nonzero(M[r:, c])[0]
        if len(nz) == 0:
            continue
        p = r + nz[0]
        if p != r:
            M[[r, p]] = M[[p, r]]
        others = np.nonzero(M[:, c])[0]
        others = others[others != r]
        if len(others):
            M[others] ^= M[r]
        piv.append(c)
        r += 1
    return M[:r], piv


def f2_rank(M):
    if len(M) == 0:
        return 0
    return len(f2_rref(M)[1])


def f2_nullspace(M, n=None):
    """Basis (rows) of {x : M x = 0} over F2."""
    M = np.array(M, dtype=np.uint8)
    if M.size == 0:
        return np.eye(n, dtype=np.uint8)
    n = M.shape[1]
    Rm, piv = f2_rref(M)
    free = [c for c in range(n) if c not in set(piv)]
    basis = []
    for f in free:
        v = np.zeros(n, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if Rm[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis, dtype=np.uint8).reshape(len(basis), n)


def f2_in_rowspace(Rref, piv, v):
    """Is v in the row space of the RREF matrix Rref (pivots piv)?"""
    v = np.array(v, dtype=np.uint8) & 1
    w = v.copy()
    for i, p in enumerate(piv):
        if w[p]:
            w ^= Rref[i]
    return not w.any()


def f2_solve(A, b):
    """Solve x A = b (x row vector, A rows = generators). Returns x or None."""
    A = np.array(A, dtype=np.uint8)
    m, n = A.shape
    aug = np.concatenate([A, np.eye(m, dtype=np.uint8)], axis=1)
    Rm, piv = f2_rref(aug)
    w = np.concatenate([np.array(b, dtype=np.uint8) & 1, np.zeros(m, dtype=np.uint8)])
    for i, p in enumerate(piv):
        if p >= n:
            break
        if w[p]:
            w ^= Rm[i]
    if w[:n].any():
        return None
    return w[n:]




import itertools
from collections import Counter, defaultdict

ARITY = {'INIT_X': 1, 'INIT_Z': 1, 'MEAS_X': 1, 'MEAS_Z': 1, 'H': 1, 'X': 1, 'Z': 1, 'S': 1, 'SDG': 1,
         'T': 1, 'TDG': 1, 'CNOT': 2, 'CZ': 2, 'CS': 2, 'CSDG': 2, 'CCZ': 3}
NONCLIFFORD = {'T', 'TDG', 'CS', 'CSDG', 'CCZ'}

In [ ]:
# ------------------------------------------------------------------ minimum-weight logical (MILP)
def min_weight_odd(Hc, lvec, cols):
    """min |x| over x in F2^cols with Hc x = 0 and lvec . x = 1 (exact MILP, HiGHS)."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    H = np.array(Hc, dtype=float)[:, cols]
    H = H[np.any(H, axis=1)]
    l = np.array(lvec, dtype=float)[cols]
    m, n = H.shape
    # variables: x (n, binary), y (m, integer), z (1, integer)
    nv = n + m + 1
    c = np.concatenate([np.ones(n), np.zeros(m + 1)])
    A1 = np.concatenate([H, -2 * np.eye(m), np.zeros((m, 1))], axis=1)
    A2 = np.concatenate([l[None, :], np.zeros((1, m)), -2 * np.ones((1, 1))], axis=1)
    cons = [LinearConstraint(A1, 0, 0), LinearConstraint(A2, 1, 1)]
    integrality = np.ones(nv)
    ub = np.concatenate([np.ones(n), np.full(m + 1, np.inf)])
    res = milp(c, constraints=cons, integrality=integrality, bounds=Bounds(np.zeros(nv), ub))
    assert res.status == 0, res.message
    x = np.round(res.x[:n]).astype(np.uint8)
    out = np.zeros(len(lvec), dtype=np.uint8)
    out[cols] = x
    return int(x.sum()), out

In [ ]:
# ------------------------------------------------------------------ gadgets
def pauli_check_gadget(anc, xq=(), zq=()):
    """Measure X(xq) Z(zq) with ancilla anc (standard CNOT circuit)."""
    g = [('INIT_X', anc)]
    g += [('CNOT', anc, q) for q in sorted(xq)]           # ancilla controls -> X on data
    g += [('CZ', anc, q) for q in sorted(zq)]             # controlled Z on data
    g += [('MEAS_X', anc)]
    return g


def z_check_gadget(anc, zq):
    g = [('INIT_Z', anc)] + [('CNOT', q, anc) for q in sorted(zq)] + [('MEAS_Z', anc)]
    return g


def clifford_x_gadget(anc, parts):
    """Measure a product of X-type Clifford operators S = S_1 S_2 ... with
    S_k = X(s_k) prod CZ(pairs_k): ancilla |+>, controlled-S (= CNOTs from anc and
    CCZ(anc,i,j) for each CZ pair; rightmost factor first), measure X.
    parts: list of (xqubits, czpairs) in operator order S_1 S_2 ..."""
    g = [('INIT_X', anc)]
    for xq, pairs in reversed(parts):
        g += [('CCZ', anc, i, j) for (i, j) in pairs]      # CZ part acts first within S_k
        g += [('CNOT', anc, q) for q in sorted(xq)]
    g += [('MEAS_X', anc)]
    return g

In [ ]:
# ------------------------------------------------------------------ scheduling
def asap_layers(gates):
    """Greedy ASAP layering of a gate list respecting qubit order (no reordering of gates that
    share a qubit).  Returns list of layers (lists of gates)."""
    ready = defaultdict(int)
    layers = []
    for gt in gates:
        qs = gt[1:]
        t = max(ready[q] for q in qs)
        while len(layers) <= t:
            layers.append([])
        layers[t].append(gt)
        for q in qs:
            ready[q] = t + 1
    return layers


def gadget_support(gad):
    s = set()
    for gt in gad:
        s |= set(gt[1:])
    return s


def disjoint_groups(gadgets):
    """Greedy colouring: partition gadgets so that gadgets in a group touch disjoint qubits.
    Measurements within a group are then exactly simultaneous and the sequential-group
    schedule is valid for commuting (on the relevant subspace) generators."""
    groups = []          # list of (used_qubits, [gadgets])
    order = sorted(range(len(gadgets)), key=lambda i: -len(gadget_support(gadgets[i])))
    for i in order:
        sup = gadget_support(gadgets[i])
        for used, members in groups:
            if not (used & sup):
                used |= sup
                members.append(gadgets[i])
                break
        else:
            groups.append((set(sup), [gadgets[i]]))
    return [m for _, m in groups]


def layer_duration(layer, dur):
    return max(dur[g[0]] for g in layer)


def circuit_time(layers, dur):
    return sum(layer_duration(l, dur) for l in layers)


def round_schedule(gadgets, dur):
    """Upper-bound schedule of one round: disjoint groups executed sequentially, each group
    ASAP-layered.  Returns dict with depth (layers), time (weighted by dur), groups."""
    groups = disjoint_groups(gadgets)
    depth = 0
    time = 0
    for grp in groups:
        gl = asap_layers([gt for gad in grp for gt in gad])
        depth += len(gl)
        time += circuit_time(gl, dur)
    return {'groups': len(groups), 'depth_ub': depth, 'time_ub': time}


def round_lower_bound(gadgets, dur):
    """Lower bound on the duration of one round: every qubit executes its gates sequentially,
    so time >= max_q sum of durations of gates acting on q."""
    busy = defaultdict(float)
    cnt = defaultdict(int)
    for gad in gadgets:
        for gt in gad:
            for q in gt[1:]:
                busy[q] += dur[gt[0]]
                cnt[q] += 1
    return {'depth_lb': max(cnt.values()), 'time_lb': max(busy.values())}


def gate_counts(gates):
    c = Counter(g[0] for g in gates)
    return dict(c)

In [ ]:
# ------------------------------------------------------------------ decompositions
# Explicit Clifford+T decompositions (verified numerically in the validation notebook).
def ccz_to_clifford_t(a, b, c):
    """CCZ with 7 T/T^dag and 6 CNOT (standard Toffoli decomposition, Nielsen & Chuang Fig. 4.9,
    with the target Hadamards removed)."""
    return [('CNOT', b, c), ('TDG', c), ('CNOT', a, c), ('T', c), ('CNOT', b, c), ('TDG', c),
            ('CNOT', a, c), ('T', b), ('T', c), ('CNOT', a, b), ('T', a), ('TDG', b), ('CNOT', a, b)]


def cs_to_clifford_t(a, b):
    """CS = diag(1,1,1,i) = T_a T_b CNOT(a,b) T^dag_b CNOT(a,b)."""
    return [('T', a), ('T', b), ('CNOT', a, b), ('TDG', b), ('CNOT', a, b)]


def csdg_to_clifford_t(a, b):
    return [('TDG', a), ('TDG', b), ('CNOT', a, b), ('T', b), ('CNOT', a, b)]


def decompose(gates):
    out = []
    for g in gates:
        if g[0] == 'CCZ':
            out += ccz_to_clifford_t(*g[1:])
        elif g[0] == 'CS':
            out += cs_to_clifford_t(*g[1:])
        elif g[0] == 'CSDG':
            out += csdg_to_clifford_t(*g[1:])
        else:
            out.append(g)
    return out


DUR_UNIT = {k: 1 for k in ARITY}            # every primitive = 1 time unit (ASSUMPTION, parametrised)

In [ ]:
# ------------------------------------------------- commuting-class scheduler
DIAG_GATES = {'CZ', 'CCZ', 'CS', 'CSDG', 'T', 'TDG', 'S', 'SDG', 'Z'}


def gate_roles(gt):
    """Per-qubit role of an entangling/1q unitary: 'Z' (diagonal in Z basis), 'X' (X-type)."""
    name = gt[0]
    if name in DIAG_GATES:
        return {q: 'Z' for q in gt[1:]}
    if name == 'CNOT':
        return {gt[1]: 'Z', gt[2]: 'X'}
    if name == 'X':
        return {gt[1]: 'X'}
    return {q: '?' for q in gt[1:]}      # H, INIT, MEAS: never shared between gadgets


def gadget_roles(gad):
    roles = {}
    for gt in gad:
        for q, r in gate_roles(gt).items():
            if roles.get(q, r) != r:
                roles[q] = '?'
            else:
                roles[q] = r
    return roles


def commuting_classes(gadgets, n_orders=6, seed=0):
    """Best (fewest classes) of several greedy colourings (different gadget orders)."""
    import random
    best = None
    orders = [list(range(len(gadgets)))]
    orders.append(sorted(orders[0], key=lambda i: tuple(sorted(set(gadget_roles(gadgets[i]).values())))))
    rnd = random.Random(seed)
    for _ in range(n_orders):
        o = orders[0][:]
        rnd.shuffle(o)
        orders.append(o)
    for o in orders:
        cl = _commuting_classes_greedy([gadgets[i] for i in o])
        if best is None or len(cl) < len(best):
            best = cl
    return best


def _commuting_classes_greedy(gadgets):
    """Greedy colouring of the gadget conflict graph.  Two gadgets are compatible if on every
    shared qubit both act with the same role ('Z' or 'X'); then all their gates commute and the
    joint circuit is equivalent to measuring them one after another (sufficient condition)."""
    roles = [gadget_roles(g) for g in gadgets]
    classes = []     # list of (role_map, [idx])
    for i, rm in enumerate(roles):
        placed = False
        for cmap, members in classes:
            ok = all(cmap.get(q, r) == r and r != '?' for q, r in rm.items() if q in cmap)
            if ok:
                for q, r in rm.items():
                    cmap[q] = r
                members.append(i)
                placed = True
                break
        if not placed:
            classes.append((dict(rm), [i]))
    return [[gadgets[i] for i in m] for _, m in classes]


def class_layers(gadgets):
    """Layering of one commuting class: init layer, greedy layering of the (mutually commuting)
    entangling gates, measurement layer."""
    inits = [g for gad in gadgets for g in gad if g[0].startswith('INIT')]
    meas = [g for gad in gadgets for g in gad if g[0].startswith('MEAS')]
    body = [g for gad in gadgets for g in gad if not (g[0].startswith('INIT') or g[0].startswith('MEAS'))]
    # greedy: most-constrained gates first
    body = sorted(body, key=lambda g: -len(g))
    layers = []
    busy = []
    for g in body:
        qs = set(g[1:])
        for li, b in enumerate(busy):
            if not (b & qs) and layers[li][0][0] == g[0]:     # keep one gate type per layer
                b |= qs
                layers[li].append(g)
                break
        else:
            busy.append(set(qs))
            layers.append([g])
    return ([inits] if inits else []) + layers + ([meas] if meas else [])


def round_schedule_classes(gadgets, dur):
    classes = commuting_classes(gadgets)
    depth = 0
    time = 0
    for c in classes:
        ls = class_layers(c)
        depth += len(ls)
        time += circuit_time(ls, dur)
    return {'classes': len(classes), 'depth_ub': depth, 'time_ub': time}


import math
from collections import Counter

SOURCE_LIT = 'literature'
SOURCE_COUNT = 'counted (explicit circuit)'
SOURCE_ASSUME = 'assumption (parameter)'
SOURCE_UNRES = 'unresolved (excluded / illustrative)'

DEFAULT_PARAMS = dict(
    kappa0=1.0,      # FSC |+_L> preparation rounds = ceil(kappa0*d)   [ASSUMPTION; Kobayashi: not specified]
    kappa_in=1.0,    # switch-in rounds            = ceil(kappa_in*d)  [ASSUMPTION; Kobayashi: O(d)]
    kappa_out=1.0,   # FSC EC rounds after return  = ceil(kappa_out*d) [ASSUMPTION; Kobayashi: O(d)]
    tau_rounds=1.0,  # Litinski: 1 time step = tau_rounds*d code cycles [LITERATURE: d cycles]
    t_inj_FT=6,      # Litinski App. A (Li's injection): 3 + 3 code cycles, d-independent [LITERATURE]
    t_inj_NL=1,      # noiseless-minimal injection: 1 round [ASSUMPTION]
)


def rounds_for(model, kappa, d):
    if model == 'NL_min':
        return 1
    return int(math.ceil(kappa * d))

In [ ]:
# ----------------------------------------------------------------- summaries
def summarize(rows):
    tot = Counter()
    for r in rows:
        for k, v in r['gates'].items():
            tot[k] += v
    peak = max(r['qubits'] for r in rows)
    rounds = sum(r['rounds'] for r in rows)
    tl = sum(r['time_lb'] for r in rows if r['time_lb'] is not None)
    tu = sum(r['time_ub'] for r in rows if r['time_ub'] is not None)
    v_round = sum(r['qubits'] * r['rounds'] for r in rows)
    v_tl = sum(r['qubits'] * r['time_lb'] for r in rows if r['time_lb'] is not None)
    v_tu = sum(r['qubits'] * r['time_ub'] for r in rows if r['time_ub'] is not None)
    nonclif = {k: v for k, v in tot.items() if k in NONCLIFFORD}
    tcount = 7 * tot.get('CCZ', 0) + 3 * (tot.get('CS', 0) + tot.get('CSDG', 0)) + tot.get('T', 0) + tot.get('TDG', 0)
    return dict(peak_qubits=peak, rounds=rounds, time_lb=tl, time_ub=tu, V_qubit_round=v_round,
                V_qubit_time_lb=v_tl, V_qubit_time_ub=v_tu, V_peak_x_rounds_UB=peak * rounds,
                gates=dict(tot), nonclifford=nonclif, T_count_if_decomposed=tcount)


def uncounted_costs(protocol, comparison):
    """Costs that are known to exist but are not included in the totals (listed, never summed as 0)."""
    out = [('classical decoding / control latency', 'both comparisons, both protocols')]
    if protocol == 'Kobayashi_D4':
        out.append(('JIT-decoder classical processing', 'not modelled'))
        if comparison == 'B':
            out.append(('FSC -> standard rotated surface-code patch conversion', 'UNRESOLVED: no protocol constructed'))
    else:
        out.append(('twist-based Z_m (x) Y_0 seam checks between |m> and |0>', 'O(d) measurement qubits, not constructed'))
    return out

In [ ]:
# ------------------------------------------------------------------ table helpers
def jsonify_columns(df, cols=('gates', 'nonclifford')):
    """Store dict-valued columns as JSON strings (same format as the saved CSV files)."""
    import json
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda g: g if isinstance(g, str) else json.dumps({k: float(v) for k, v in g.items()}))
    return df


def identify_polynomial(values, npts=4):
    """Interpolate a polynomial in d through the first npts points (degree <= npts-1) and report
    whether it reproduces ALL given points exactly.  values: dict d -> integer.  Returns (expr, ok)."""
    import sympy
    x = sympy.Symbol('d')
    ds = sorted(values)
    if len(ds) <= npts:
        return None, False
    p = sympy.interpolate([(dd, int(round(values[dd]))) for dd in ds[:npts]], x)
    ok = all(sympy.nsimplify(p.subs(x, dd) - int(round(values[dd]))) == 0 for dd in ds)
    return sympy.expand(p), ok

### D4 ライブラリ（境界付き D4 符号・transversal $U$・code switching・疎状態シミュレータ・資源モデル）

In [ ]:
"""d4_lib: D4 Clifford stabilizer code (Kobayashi-Zhu-Hsin) on the bounded square lattice, transversal U, code switching, resource model."""
import itertools, cmath
import numpy as np
from collections import Counter

R, G, B = 0, 1, 2
CNAME = "rgb"

In [ ]:
# ------------------------------------------------- phase-shift operators
class PSOp:
    """Operator  O |x> = (-1)^{f(x)} |x + s>  on n qubits (computational basis x in F2^n).

    f(x) = const + sum_{i in lin} x_i + sum_{(i,j) in quad} x_i x_j   (mod 2), i<j.
    X-type Clifford stabilizers of the D4 code, Pauli X/Z strings and products
    thereof all have this form.  Composition rule (A o B)|x> = A B |x>:
        s = s_A + s_B,   f(x) = f_B(x) + f_A(x + s_B).
    """
    __slots__ = ("s", "lin", "quad", "const")

    def __init__(self, s=frozenset(), lin=frozenset(), quad=frozenset(), const=0):
        self.s = frozenset(s)
        self.lin = frozenset(lin)
        self.quad = frozenset(tuple(sorted(p)) for p in quad)
        self.const = const & 1

    def shifted_phase(self, t):
        """Return (lin, quad, const) of x -> f(x + t)."""
        lin = set(self.lin)
        const = self.const
        for i in self.lin:
            if i in t:
                const ^= 1
        for (i, j) in self.quad:
            ti, tj = i in t, j in t
            if ti and tj:
                const ^= 1
            if tj:
                lin ^= {i}
            if ti:
                lin ^= {j}
        return frozenset(lin), self.quad, const

    def __matmul__(self, other):
        lA, qA, cA = self.shifted_phase(other.s)
        return PSOp(self.s ^ other.s, lA ^ other.lin, qA ^ other.quad, cA ^ other.const)

    def phase_at(self, x):
        """f(x) for x given as a set/frozenset of 1-positions."""
        v = self.const
        for i in self.lin:
            if i in x:
                v ^= 1
        for (i, j) in self.quad:
            if i in x and j in x:
                v ^= 1
        return v

    def apply(self, x):
        """O|x> = sign |x'> ; returns (x', sign_bit)."""
        return frozenset(x) ^ self.s, self.phase_at(x)

    def support(self):
        sup = set(self.s) | set(self.lin)
        for p in self.quad:
            sup |= set(p)
        return sup


def commutator_affine(A, B):
    """A B = (-1)^{g(x)} B A on |x>.  Returns g as (lin, const) -- quadratic terms cancel."""
    AB = A @ B
    BA = B @ A
    assert AB.s == BA.s
    q = AB.quad ^ BA.quad
    assert not q, "commutator has quadratic part (should not happen)"
    return AB.lin ^ BA.lin, AB.const ^ BA.const

In [ ]:
# ------------------------------------------------------------- the lattice
class D4Lattice:
    def __init__(self, L):
        assert L >= 1
        self.L = L
        self.edges = []          # (kind, x, y), kind 'h' or 'v'
        self.eidx = {}
        for y in range(L + 1):
            for x in range(L):
                self._add(('h', x, y))
        for x in range(L + 1):
            for y in range(L):
                self._add(('v', x, y))
        self.E = len(self.edges)
        self.n = 3 * self.E
        self.vertices = [(x, y) for y in range(L + 1) for x in range(L + 1)]
        self.plaquettes = [(x, y) for y in range(L) for x in range(L)]

    def _add(self, e):
        self.eidx[e] = len(self.edges)
        self.edges.append(e)

    def q(self, e, c):
        if not isinstance(e, int):
            e = self.eidx[e]
        return 3 * e + c

    # --- geometry --------------------------------------------------------
    def endpoints(self, e):
        k, x, y = self.edges[e] if isinstance(e, int) else e
        return ((x, y), (x + 1, y)) if k == 'h' else ((x, y), (x, y + 1))

    def incident(self, v):
        x, y = v
        out = []
        for e in [('h', x, y), ('h', x - 1, y), ('v', x, y), ('v', x, y - 1)]:
            if e in self.eidx:
                out.append(self.eidx[e])
        return out

    def plaq_edges(self, p):
        """(e01, e13, e02, e23) of plaquette p=(x,y)."""
        x, y = p
        ei = self.eidx
        return ei[('h', x, y)], ei[('v', x + 1, y)], ei[('v', x, y)], ei[('h', x, y + 1)]

    def plaq_boundary(self, p):
        return list(self.plaq_edges(p))

    def has_plaq(self, x, y):
        return 0 <= x < self.L and 0 <= y < self.L

    def boundary_of_edge(self, e):
        k, x, y = self.edges[e]
        L = self.L
        if k == 'v' and x == 0:
            return 'Lr'
        if k == 'h' and y == L:
            return 'Lb'
        if k == 'h' and y == 0:
            return 'Lrb'
        if k == 'v' and x == L:
            return 'Lrb'
        return None

    def boundary_edges(self, name):
        return [e for e in range(self.E) if self.boundary_of_edge(e) == name]

    # --- Z-type checks (rows of H_Z) --------------------------------------
    def z_checks(self, colours=(R, G, B)):
        """List of (label, set_of_qubits) Z-type stabilizer generators.
        Plaquettes for every colour + single-edge boundary conditions (Fig. 4 / eq. (7))."""
        rows = []
        for p in self.plaquettes:
            for c in colours:
                rows.append((('plaq', p, CNAME[c]), {self.q(e, c) for e in self.plaq_boundary(p)}))
        for e in range(self.E):
            bd = self.boundary_of_edge(e)
            if bd == 'Lr' and R in colours:
                rows.append((('bdZ', e, 'r'), {self.q(e, R)}))
            elif bd == 'Lb' and B in colours:
                rows.append((('bdZ', e, 'b'), {self.q(e, B)}))
            elif bd == 'Lrb':
                if R in colours and B in colours:
                    rows.append((('bdZ', e, 'rb'), {self.q(e, R), self.q(e, B)}))
                if G in colours:
                    rows.append((('bdZ', e, 'g'), {self.q(e, G)}))
        return rows

    # --- X-type Clifford stabilizers ----------------------------------------
    def cz_pairs(self, v, c):
        """CZ pairs (qubit, qubit) of the dressing of S^c_{X;v} (SM eq. for S_X, cup products
        evaluated with the hypercube formula; plaquettes outside the lattice are dropped)."""
        x, y = v
        pairs = []
        if c == R:      # int v u e' u e'' : v = vertex 0 of p(x,y);   CZ^{g,b}
            if self.has_plaq(x, y):
                e01, e13, e02, e23 = self.plaq_edges((x, y))
                pairs += [(self.q(e01, G), self.q(e13, B)), (self.q(e02, G), self.q(e23, B))]
        elif c == G:    # int e'' u v u e' : v = vertex 1 of p(x-1,y), vertex 2 of p(x,y-1); CZ^{r,b}
            if self.has_plaq(x - 1, y):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y))
                pairs.append((self.q(e01, R), self.q(e13, B)))
            if self.has_plaq(x, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x, y - 1))
                pairs.append((self.q(e02, R), self.q(e23, B)))
        else:           # int e' u e'' u v : v = vertex 3 of p(x-1,y-1); CZ^{r,g}
            if self.has_plaq(x - 1, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y - 1))
                pairs += [(self.q(e01, R), self.q(e13, G)), (self.q(e02, R), self.q(e23, G))]
        return pairs

    def SX(self, v, c, dressed=True):
        s = {self.q(e, c) for e in self.incident(v)}
        quad = self.cz_pairs(v, c) if dressed else []
        # CZ applied first, then X (S = (prod X)(prod CZ)); X and CZ act on disjoint colours.
        return PSOp(s=s, quad=quad)

    def vertex_x_types(self, v, colours=(R, G, B)):
        """Allowed colour combinations at vertex v: the F2 span of colour sets whose
        X-part commutes with the boundary Z conditions.  Returns a list of colour tuples
        (a basis), derived automatically from the boundary Z checks."""
        bdrows = [rows for (lab, rows) in self.z_checks(colours) if lab[0] == 'bdZ']
        inc = self.incident(v)
        cand = []
        for k in range(1, len(colours) + 1):
            for combo in itertools.combinations(colours, k):
                s = {self.q(e, c) for e in inc for c in combo}
                if all(len(s & rw) % 2 == 0 for rw in bdrows):
                    cand.append(combo)
        # reduce to an F2 basis over colour space, preferring single colours
        basis, vecs = [], []
        for combo in sorted(cand, key=len):
            vec = np.zeros(3, dtype=np.uint8)
            vec[list(combo)] = 1
            if f2_rank(np.array(vecs + [vec])) > len(vecs):
                vecs.append(vec)
                basis.append(combo)
        return basis

    def x_generators(self, colours=(R, G, B), dressed=True):
        """List of (label, PSOp).  At a vertex where only a product of colours is allowed
        (L_rb), the generator is the ordered product S^r S^b (Fig. 4 shows S^r S^b there)."""
        gens = []
        for v in self.vertices:
            for combo in self.vertex_x_types(v, colours):
                op = PSOp()
                for c in combo:
                    op = op @ self.SX(v, c, dressed)
                gens.append((('X', v, ''.join(CNAME[c] for c in combo)), op))
        return gens

    def vec(self, qubits):
        v = np.zeros(self.n, dtype=np.uint8)
        v[list(qubits)] = 1
        return v

In [ ]:
# ---------------------------------------------------------- code data
class CodeData:
    """Linear-algebra data of a (D4 or folded-surface) code on a lattice."""

    def __init__(self, lat, colours=(R, G, B), dressed=True):
        self.lat = lat
        self.colours = tuple(colours)
        self.zlabels, zr = zip(*lat.z_checks(colours))
        self.HZ = np.array([lat.vec(r) for r in zr], dtype=np.uint8)
        self.xgens = lat.x_generators(colours, dressed)
        self.S = np.array([lat.vec(op.s) for _, op in self.xgens], dtype=np.uint8)
        self.cols = sorted({lat.q(e, c) for e in range(lat.E) for c in colours})
        self.HZr, self.HZpiv = f2_rref(self.HZ)
        self.Sr, self.Spiv = f2_rref(self.S)
        # logical representatives
        CZ = f2_nullspace(self.HZ[:, self.cols])
        self.t_log = None
        for row in CZ:
            t = np.zeros(lat.n, dtype=np.uint8)
            t[self.cols] = row
            if not f2_in_rowspace(self.Sr, self.Spiv, t):
                self.t_log = t
                break
        KS = f2_nullspace(self.S[:, self.cols])
        self.z_log = None
        for row in KS:
            u = np.zeros(lat.n, dtype=np.uint8)
            u[self.cols] = row
            if not f2_in_rowspace(self.HZr, self.HZpiv, u):
                self.z_log = u
                break
        self.k = len(self.cols) - f2_rank(self.HZ[:, self.cols]) - f2_rank(self.S[:, self.cols])


def distances(cd):
    """(d_Z, d_X_shift) : min weight of a logical Z-type Pauli, and min weight of an
    X-shift lying in C_Z but outside the stabilizer shift span."""
    dz, uz = min_weight_odd(cd.S, cd.t_log, cd.cols)
    dx, tx = min_weight_odd(cd.HZ, cd.z_log, cd.cols)
    return dz, dx, uz, tx

In [ ]:
# ------------------------------------------- transversal automorphism U = W V
class DiagPhase:
    """Diagonal unitary |x> -> omega^{w(x)} |x>, omega = exp(i pi/4), w integer mod 8.
    w(x) = sum_k c_k * prod_{i in mono_k} x_i  (monomials of degree <= 2)."""

    def __init__(self, terms=()):
        self.terms = list(terms)   # (coef mod 8, tuple_of_qubits)

    def value(self, x):
        v = 0
        for c, mono in self.terms:
            if all(i in x for i in mono):
                v += c
        return v % 8

    def support(self):
        s = set()
        for _, m in self.terms:
            s |= set(m)
        return s

    def terms_touching(self, qs):
        return DiagPhase([(c, m) for c, m in self.terms if set(m) & qs])


def W_operator(lat, cs_signs=(+1, -1), boundary_T=-1):
    """W of eq. (8):  prod_p CS^{r,b}_{e01,e13} (CS^{r,b}_{e02,e23})^dag  x  prod_{e in bdry_rb} (T^r_e)^dag.
    CS = diag(1,1,1,i) -> omega^{2 x y};  T^dag -> omega^{-x}."""
    terms = []
    for p in lat.plaquettes:
        e01, e13, e02, e23 = lat.plaq_edges(p)
        terms.append(((2 * cs_signs[0]) % 8, (lat.q(e01, R), lat.q(e13, B))))
        terms.append(((2 * cs_signs[1]) % 8, (lat.q(e02, R), lat.q(e23, B))))
    if boundary_T:
        for e in lat.boundary_edges('Lrb'):
            terms.append((boundary_T % 8, (lat.q(e, R),)))
    return DiagPhase(terms)


def V_map(lat, x):
    """V = prod_e CNOT^{(r,g)}_e CNOT^{(b,g)}_e :  x_g <- x_g + x_r + x_b  (set representation)."""
    x = set(x)
    out = set(x)
    for e in range(lat.E):
        qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
        if (qr in x) ^ (qb in x):
            out ^= {qg}
    return frozenset(out)


def V_support_map(lat, qs):
    """Qubits whose values determine (V x) restricted to qs."""
    out = set()
    for q in qs:
        e, c = divmod(q, 3)
        out.add(q)
        if c == G:
            out |= {3 * e + R, 3 * e + B}
    return out


def CZ_basis(cd):
    if not hasattr(cd, '_CZbasis'):
        nb = f2_nullspace(cd.HZ[:, cd.cols])
        full = np.zeros((nb.shape[0], cd.lat.n), dtype=np.uint8)
        full[:, cd.cols] = nb
        cd._CZbasis = full
    return cd._CZbasis


def project_CZ(cd, Q, functionals=()):
    """Exact image of C_Z under x -> (x|_Q, phi_1(x), ..., phi_k(x)) for F2-linear functionals
    phi_j (callables on a set of qubits, evaluated on basis vectors).  Returns a list of
    configurations (frozenset of Q-qubits) or, if functionals are given, (config, bits)."""
    Q = sorted(Q)
    Bs = CZ_basis(cd)
    P = Bs[:, Q]
    if functionals:
        extra = np.array([[phi(frozenset(np.nonzero(b)[0])) for phi in functionals] for b in Bs],
                         dtype=np.uint8).reshape(len(Bs), len(functionals))
        P = np.concatenate([P, extra], axis=1)
    Pr, piv = f2_rref(P)
    k = Pr.shape[0]
    assert k <= 24, f"local projection too large ({k})"
    out = []
    nq = len(Q)
    for bits in itertools.product((0, 1), repeat=k):
        v = np.zeros(P.shape[1], dtype=np.uint8)
        for b, row in zip(bits, Pr):
            if b:
                v ^= row
        cfg = frozenset(Q[i] for i in np.nonzero(v[:nq])[0])
        out.append((cfg, tuple(int(z) for z in v[nq:])) if functionals else cfg)
    return out


def check_conjugation(cd, Wop, gen_op, target_op):
    """Exhaustively check  U S U^dag == S'  on C_Z, where U = W V.
    Returns (ok, n_local_configs)."""
    lat = cd.lat
    As = V_map(lat, gen_op.s)
    if As != target_op.s:
        return False, 0
    # qubits on which the phase difference can depend
    Wt = Wop.terms_touching(set(As))
    Q = set(Wt.support()) | V_support_map(lat, gen_op.support()) | target_op.support() | set(As)
    configs = project_CZ(cd, Q)
    for x in configs:
        # omega^{ w(x+As) - w(x) + 4 f(A x) - 4 f'(x) }   (only W-terms touching As contribute)
        d = Wt.value(x ^ As) - Wt.value(x) + 4 * gen_op.phase_at(V_map(lat, x)) - 4 * target_op.phase_at(x)
        if d % 8:
            return False, len(configs)
    return True, len(configs)


def stabilizer_path_phase(cd, x, target):
    """Given x and target = x + (sum of generator shifts), apply generators to |x> and
    return the accumulated sign bit, i.e. psi(target) = (-1)^bit psi(x) for code states."""
    shifts = cd.S
    delta = cd.lat.vec(set(x) ^ set(target))
    coeff = f2_solve(shifts, delta)
    if coeff is None:
        return None
    bit = 0
    cur = frozenset(x)
    for i in np.nonzero(coeff)[0]:
        op = cd.xgens[i][1]
        cur, b = op.apply(cur)
        bit ^= b
    assert cur == frozenset(target)
    return bit


def logical_phases(cd, Wop):
    """Eigenphases (in units of pi/4) of U on |0_L>, |1_L>, assuming U preserves the code space
    (checked separately).  Uses x0 = 0 and x1 = t_log."""
    lat = cd.lat
    res = []
    for x in [frozenset(), frozenset(np.nonzero(cd.t_log)[0])]:
        Ax = V_map(lat, x)
        bit = stabilizer_path_phase(cd, x, Ax)
        assert bit is not None, "V does not map the coset to itself"
        res.append((Wop.value(Ax) + 4 * bit) % 8)
    return res

In [ ]:
# ------------------------------------------- per-configuration protocol phase
def lrb_vertices(lat):
    vs = set()
    for e in lat.boundary_edges('Lrb'):
        vs |= set(lat.endpoints(e))
    return vs


def cprime(lat, a):
    """0-cochain c' with  delta c' = a_r + a_b  and c' = 0 on L_rb vertices (BFS).
    a: set of qubits (configuration of r,b qubits).  Returns dict vertex -> bit.
    Raises if a_r + a_b is not exact relative to L_rb."""
    from collections import deque
    val = {v: 0 for v in lrb_vertices(lat)}
    dq = deque(val)
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            bit = val[u] ^ (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
            if w in val:
                if val[w] != bit:
                    raise ValueError("a_r + a_b not exact")
            else:
                val[w] = bit
                dq.append(w)
    return val


def g_phase_poly(lat, v):
    """CZ^{r,b} pairs of S^g_{X;v}  (f^g_v)."""
    return lat.cz_pairs(v, G)


def protocol_phase(lat, Wop, a, gverts):
    """Phase exponent (mod 8, units of pi/4) acquired by FSC configuration a through
    switch-in (S^g measurement) -> U -> Z^g measurement -> S^g correction -> Pauli frame,
    i.e.  F(a) = w(a) + 4 * sum_v c'_v(a) f^g_v(a)."""
    cp = cprime(lat, a)
    phi = 0
    for v in gverts:
        if cp[v]:
            for (i, j) in g_phase_poly(lat, v):
                if i in a and j in a:
                    phi ^= 1
    return (Wop.value(a) + 4 * phi) % 8


def check_protocol_invariance(lat, fsc, Wop, gverts, gen_op):
    """Exhaustive local proof that F(a + t) = F(a) (mod 8) for all a in C_FSC, t = shift of gen_op."""
    t = frozenset(gen_op.s)
    # c'(t) (linear in a)
    ct = cprime(lat, t)
    N2 = {v for v in gverts if ct[v]}
    N1 = {v for v in gverts if set(q for pr in g_phase_poly(lat, v) for q in pr) & t}
    Wt = Wop.terms_touching(set(t))
    Q = set(Wt.support()) | set(t)
    for v in N1 | N2:
        Q |= {q for pr in g_phase_poly(lat, v) for q in pr}
    # paths from a reference vertex w0 to each w in N1 (BFS on the lattice graph)
    w0 = min(N1 | N2 | {lat.endpoints(next(iter(t)) // 3)[0]})
    paths = {}
    from collections import deque
    prev = {w0: None}
    dq = deque([w0])
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            if w not in prev:
                prev[w] = (u, e)
                dq.append(w)
    for w in N1:
        es = []
        cur = w
        while prev[cur] is not None:
            u, e = prev[cur]
            es.append(e)
            cur = u
        paths[w] = es
        Q |= {lat.q(e, c) for e in es for c in (R, B)}
    beta_fn = lambda b: cprime(lat, b)[w0]          # c'_{w0}(a) is F2-linear in a
    configs = project_CZ(fsc, Q, functionals=[beta_fn])
    for a, (beta,) in configs:
        if True:
            cpa = {}
            for w in N1:
                bit = beta
                for e in paths[w]:
                    bit ^= (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
                cpa[w] = bit
            at = a ^ t
            d = Wt.value(at) - Wt.value(a)
            phi = 0
            for w in N1:
                if cpa[w]:
                    for (i, j) in g_phase_poly(lat, w):
                        phi ^= ((i in at) and (j in at)) ^ ((i in a) and (j in a))
            for w in N2:
                for (i, j) in g_phase_poly(lat, w):
                    phi ^= ((i in at) and (j in at))
            if (d + 4 * phi) % 8:
                return False, len(configs)
    return True, len(configs)


OMEGA = cmath.exp(1j * cmath.pi / 4)


class SparseState:
    def __init__(self, amps):
        self.a = {k: v for k, v in amps.items() if abs(v) > 1e-14}

    def norm2(self):
        return sum(abs(v) ** 2 for v in self.a.values())

    def normalize(self):
        n = self.norm2() ** 0.5
        self.a = {k: v / n for k, v in self.a.items()}
        return self

    def apply_psop(self, op):
        out = {}
        for x, v in self.a.items():
            y, b = op.apply(x)
            out[y] = out.get(y, 0) + (-v if b else v)
        return SparseState(out)

    def apply_diag(self, D):
        return SparseState({x: v * OMEGA ** D.value(x) for x, v in self.a.items()})

    def apply_perm(self, f):
        out = {}
        for x, v in self.a.items():
            y = f(x)
            out[y] = out.get(y, 0) + v
        return SparseState(out)

    def project(self, op, sign):
        """(1 + sign*O)/2 |psi>  (unnormalised)."""
        Opsi = self.apply_psop(op).a
        out = {}
        for k in set(self.a) | set(Opsi):
            out[k] = 0.5 * (self.a.get(k, 0) + sign * Opsi.get(k, 0))
        return SparseState(out)

    def expect(self, op):
        Opsi = self.apply_psop(op).a
        return sum(np.conj(self.a.get(k, 0)) * v for k, v in Opsi.items())

    def measure_op(self, op, rng, forced=None):
        """Projective measurement of a Hermitian involution O; returns (outcome bit, state)."""
        p0 = self.project(op, +1)
        n0 = p0.norm2()
        if forced is None:
            bit = 0 if rng.random() < n0 else 1
        else:
            bit = forced
        st = p0 if bit == 0 else self.project(op, -1)
        prob = n0 if bit == 0 else 1 - n0
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def measure_z(self, q, rng, forced=None):
        p1 = sum(abs(v) ** 2 for x, v in self.a.items() if q in x)
        bit = (1 if rng.random() < p1 else 0) if forced is None else forced
        st = SparseState({x: v for x, v in self.a.items() if (q in x) == bool(bit)})
        prob = p1 if bit else 1 - p1
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def overlap(self, other):
        return sum(np.conj(v) * other.a.get(k, 0) for k, v in self.a.items())


def coset_state(cd, rep, phases=True):
    """Code state supported on rep + span(shifts):  psi(rep)=1, psi(x+s_v)=(-1)^{f_v(x)} psi(x)."""
    amps = {frozenset(rep): 1.0}
    # breadth-first application of generators
    frontier = [frozenset(rep)]
    while frontier:
        new = []
        for x in frontier:
            for _, op in cd.xgens:
                y, b = op.apply(x)
                if y not in amps:
                    amps[y] = amps[x] * (-1 if b else 1)
                    new.append(y)
                else:
                    if phases:
                        assert abs(amps[y] - amps[x] * (-1 if b else 1)) < 1e-9, "inconsistent code state"
        frontier = new
    return SparseState(amps).normalize()




def frozen_qubits(lat):
    """Qubits fixed to |0> by single-qubit boundary Z stabilizers (L_r: r, L_b: b, L_rb: g)."""
    fz = set()
    for lab, rows in lat.z_checks():
        if lab[0] == 'bdZ' and len(rows) == 1:
            fz |= rows
    return fz


class D4Protocol:
    def __init__(self, L, prune_frozen=True):
        self.L = L
        self.lat = lat = D4Lattice(L)
        self.frozen = frozen_qubits(lat) if prune_frozen else set()
        self.prune = prune_frozen
        self.d4 = CodeData(lat)
        self.fsc = CodeData(lat, (R, B), dressed=False)
        # ancilla allocation: one ancilla per stabilizer generator label (D4 superset)
        self.anc = {}
        nxt = lat.n
        labels = [lab for lab, _ in self.d4.xgens] + list(self.d4.zlabels)
        labels += [lab for lab, _ in self.fsc.xgens] + list(self.fsc.zlabels)
        for lab in labels:
            key = self._anc_key(lab)
            if key not in self.anc and not self._is_frozen_check(lab):
                self.anc[key] = nxt
                nxt += 1
        self.gverts = [lab[1] for lab, op in self.d4.xgens if lab[2] == 'g']

    @staticmethod
    def _anc_key(lab):
        return lab

    def _is_frozen_check(self, lab):
        return lab[0] == 'bdZ' and lab[2] in ('r', 'b', 'g') and self.prune

    def data_qubits(self, colours):
        return {self.lat.q(e, c) for e in range(self.lat.E) for c in colours} - self.frozen

    # ---------------------------------------------------------- gadgets
    def _keep_pair(self, i, j):
        return i not in self.frozen and j not in self.frozen

    def x_gadget(self, lab, dressed):
        _, v, cols = lab
        parts = []
        for ch in cols:
            c = CNAME.index(ch)
            xq = {self.lat.q(e, c) for e in self.lat.incident(v)} - self.frozen
            pairs = [p for p in self.lat.cz_pairs(v, c) if self._keep_pair(*p)] if dressed else []
            parts.append((xq, pairs))
        return clifford_x_gadget(self.anc[lab], parts)

    def z_gadget(self, lab, rows):
        return z_check_gadget(self.anc[lab], set(rows) - self.frozen)

    def round_gadgets(self, code, which='all'):
        """Gadgets of one stabilizer round.  code: 'D4' or 'FSC'.
        which: 'all' | 'X' | 'Z' | 'g' (D4 S^g only)."""
        cd = self.d4 if code == 'D4' else self.fsc
        dressed = code == 'D4'
        gads = []
        if which in ('all', 'X', 'g'):
            for lab, op in cd.xgens:
                if which == 'g' and lab[2] != 'g':
                    continue
                gads.append(self.x_gadget(lab, dressed))
        if which in ('all', 'Z'):
            for lab, (l2, rows) in zip(cd.zlabels, self.lat.z_checks(cd.colours)):
                if self._is_frozen_check(lab):
                    continue
                gads.append(self.z_gadget(lab, rows))
        return gads

    # ---------------------------------------------------------- transversal U
    def U_gates(self):
        lat, fz = self.lat, self.frozen
        gates = []
        for e in range(lat.E):
            qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
            for ctrl in (qr, qb):
                if ctrl in fz:
                    continue
                if qg in fz:
                    continue   # L_rb: CNOT(r,g) CNOT(b,g) = identity on Z^r Z^b = +1 (g removed)
                gates.append(('CNOT', ctrl, qg))
        for p in lat.plaquettes:
            e01, e13, e02, e23 = lat.plaq_edges(p)
            for (a, b, name) in [(lat.q(e01, R), lat.q(e13, B), 'CS'), (lat.q(e02, R), lat.q(e23, B), 'CSDG')]:
                if a in fz or b in fz:
                    continue
                gates.append((name, a, b))
        for e in lat.boundary_edges('Lrb'):
            gates.append(('TDG', lat.q(e, R)))
        return gates

    # ---------------------------------------------------------- switch-out
    def switch_out_gates(self, worst_case=True):
        lat = self.lat
        gq = sorted(self.data_qubits((G,)))
        gates = [('MEAS_Z', q) for q in gq]
        cz = []
        for v in self.gverts:
            for p in lat.cz_pairs(v, G):
                if self._keep_pair(*p):
                    cz.append(('CZ',) + tuple(p))
        return gates, cz      # cz: worst-case correction (all vertices in kappa)

    def qubit_sets(self):
        fsc_data = self.data_qubits((R, B))
        d4_data = self.data_qubits((R, G, B))
        fsc_anc = {self.anc[l] for l, _ in self.fsc.xgens} | {self.anc[l] for l in self.fsc.zlabels
                                                              if not self._is_frozen_check(l)}
        d4_anc = {self.anc[l] for l, _ in self.d4.xgens} | {self.anc[l] for l in self.d4.zlabels
                                                            if not self._is_frozen_check(l)}
        return dict(fsc_data=fsc_data, d4_data=d4_data, fsc_anc=fsc_anc, d4_anc=d4_anc)

In [ ]:
# ----------------------------------------------------------------- Kobayashi (D4)
class D4CostData:
    """All d-dependent counts of the D4 protocol, computed once per L from explicit circuits."""

    def __init__(self, L, dur=None, decomposed=False):
        dur = dur or DUR_UNIT
        P = D4Protocol(L)
        self.L = L
        qs = P.qubit_sets()
        self.n = {k: len(v) for k, v in qs.items()}
        self.n_total = len(qs['d4_data'] | qs['d4_anc'])
        dec = decompose if decomposed else (lambda g: g)
        gad = {
            'D4': [dec(g) for g in P.round_gadgets('D4')],
            'D4g': [dec(g) for g in P.round_gadgets('D4', 'g')],
            'FSC': P.round_gadgets('FSC'),
        }
        self.round = {}
        for k, g in gad.items():
            ub = round_schedule_classes(g, dur)
            lb = round_lower_bound(g, dur)
            self.round[k] = dict(time_lb=lb['time_lb'], time_ub=ub['time_ub'], depth_lb=lb['depth_lb'],
                                 depth_ub=ub['depth_ub'], classes=ub['classes'],
                                 gates=Counter(gate_counts([x for gg in g for x in gg])),
                                 n_anc=len(g))
        self.n_anc_g = len(gad['D4g'])
        U = dec(P.U_gates())
        Ul = class_layers([U])              # all U gates mutually commute (validated)
        self.U = dict(gates=Counter(gate_counts(U)), time=circuit_time(Ul, dur), depth=len(Ul))
        meas, cz = P.switch_out_gates()
        czl = class_layers([cz]) if cz else []
        ml = [meas]
        self.out = dict(gates_worst=Counter(gate_counts(meas + cz)),
                        gates_expected=Counter({**gate_counts(meas), 'CZ': len(cz) / 2}),
                        time=circuit_time(ml + czl, dur), depth=len(ml) + len(czl))
        self.x_corr = dict(time=dur['X'], gates_worst=Counter({'X': self.n['fsc_data']}))
        self.init = Counter({'INIT_X': self.n['fsc_data'], 'INIT_Z': len(qs['d4_data']) - self.n['fsc_data']})


def d4_stages(cd, model, params=DEFAULT_PARAMS, comparison='A', data_only=False):
    d = cd.L
    n = cd.n
    fsc_q = n['fsc_data'] + (0 if data_only else n['fsc_anc'])
    d4_q = n['d4_data'] + (0 if data_only else n['d4_anc'])
    rows = []

    def add(stage, qubits, rounds, rkey, extra_time, gates, source, note=''):
        rt = cd.round[rkey] if rkey else dict(time_lb=0, time_ub=0)
        rows.append(dict(protocol='Kobayashi_D4', comparison=comparison, model=model, d=d, stage=stage,
                         qubits=qubits, rounds=rounds,
                         time_lb=rounds * rt['time_lb'] + extra_time,
                         time_ub=rounds * rt['time_ub'] + extra_time,
                         gates=dict(gates), source=source, note=note))

    r0 = rounds_for(model, params['kappa0'], d)
    g0 = Counter(cd.init)
    for k, v in cd.round['FSC']['gates'].items():
        g0[k] += v * r0
    g0['X'] += cd.x_corr['gates_worst']['X']
    add('S0 FSC |+_L> preparation', fsc_q, r0, 'FSC', cd.x_corr['time'], g0,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'kappa0*d rounds (not specified in Kobayashi); X-correction worst case counted')
    if model == 'NL_min':
        r1, key = 1, 'D4g'
        q1 = n['d4_data'] + (0 if data_only else cd.n_anc_g)
        note1 = 'noiseless minimum: one round of S^g_X measurements only'
    else:
        r1, key = rounds_for(model, params['kappa_in'], d), 'D4'
        q1 = d4_q
        note1 = 'O(d) rounds of all D4 stabilizers (Kobayashi step 2); kappa_in*d assumed'
    g1 = Counter()
    for k, v in cd.round[key]['gates'].items():
        g1[k] += v * r1
    add('S1 switch-in (gauging) to D4', q1, r1, key, 0, g1,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT, note1)
    add('S2 transversal U = W V (logical T^dag)', n['d4_data'], 0, None, cd.U['time'], cd.U['gates'],
        SOURCE_COUNT, 'constant depth; gates from eq. (8)')
    add('S3 switch-out: Z^g meas + S^g correction', n['d4_data'], 0, None, cd.out['time'],
        cd.out['gates_worst'], SOURCE_COUNT, 'CZ correction counted worst case (expected = half)')
    r4 = 0 if model == 'NL_min' else rounds_for(model, params['kappa_out'], d)
    g4 = Counter()
    for k, v in cd.round['FSC']['gates'].items():
        g4[k] += v * r4
    add('S4 FSC error-correction rounds', fsc_q, r4, 'FSC', 0, g4,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'O(d) rounds (Kobayashi step 4); kappa_out*d assumed; 0 in the noiseless minimum')
    add('S5 logical X frame: T^dag|+> -> T|+>', 0, 0, None, 0, Counter(), SOURCE_COUNT,
        'Pauli frame update, no physical cost')
    # NOTE (comparison B): the FSC -> standard rotated-patch conversion is UNRESOLVED.  It is deliberately
    # NOT added as a zero-cost stage; it is reported separately by uncounted_costs().
    return rows



def d4_conversion_illustrative(cd, model, params=DEFAULT_PARAMS, data_only=False):
    """ILLUSTRATIVE ONLY: conversion modelled as one lattice-surgery time step (tau) during which the
    FSC plus 2 tiles (target patch + ancilla strip) are active.  Not taken from literature."""
    d = cd.L
    r = rounds_for(model, params['tau_rounds'], d)
    q = cd.n['fsc_data'] + (0 if data_only else cd.n['fsc_anc']) + 2 * (d * d if data_only else 2 * d * d)
    return dict(protocol='Kobayashi_D4', comparison='B_illustrative', model=model, d=d,
                stage='S6 conversion (ILLUSTRATIVE: 1 tau lattice surgery)', qubits=q, rounds=r,
                time_lb=None, time_ub=None, gates={}, source=SOURCE_UNRES,
                note='illustrative assumption, not a validated protocol')

In [ ]:
# ---------------------------------------------------------- consistency checks of a code
def code_consistency(cd):
    """Exact F2 checks of a (possibly non-commuting) Clifford stabilizer code on its Z-subspace C_Z:
      shift_in_CZ : every X-type generator preserves C_Z (its X part commutes with all Z checks)
      noncommuting_pairs : generator pairs that do NOT commute on C_Z
      bad_squares : generators with S^2 != 1 on C_Z
      k : number of logical qubits = dim C_Z - rank(shifts)   (generators have independent shifts)"""
    lat = cd.lat
    HZ, S = cd.HZ, cd.S
    res = {}
    res['shift_in_CZ'] = not ((HZ.astype(int) @ S.T.astype(int)) % 2).any()
    supp = [op.support() for _, op in cd.xgens]
    bad = []
    for i in range(len(cd.xgens)):
        for j in range(i + 1, len(cd.xgens)):
            if not (supp[i] & supp[j]):
                continue
            lin, const = commutator_affine(cd.xgens[i][1], cd.xgens[j][1])
            if const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(lin)):
                bad.append((cd.xgens[i][0], cd.xgens[j][0]))
    res['noncommuting_pairs'] = bad
    sq = []
    for lab, op in cd.xgens:
        s2 = op @ op
        if s2.s or s2.quad or s2.const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(s2.lin)):
            sq.append(lab)
    res['bad_squares'] = sq
    res['n_qubits'] = len(cd.cols)
    res['dim_CZ'] = len(cd.cols) - f2_rank(HZ[:, cd.cols])
    res['rank_shifts'] = f2_rank(S[:, cd.cols])
    res['n_xgens'] = len(cd.xgens)
    res['k'] = res['dim_CZ'] - res['rank_shifts']
    res['shifts_independent'] = res['rank_shifts'] == res['n_xgens']
    return res


def U_target(lat, lab):
    """Image of generator `lab` under U predicted by SM eq. (permutingstabilizers):
    S^r -> S^r S^g, S^g -> S^g, S^b -> S^g S^b, and (boundary L_rb) S^r S^b -> S^r S^b."""
    _, v, cols = lab
    if cols == 'rb':
        return lat.SX(v, R) @ lat.SX(v, B)
    op = PSOp()
    for ch in cols:
        c = CNAME.index(ch)
        for cc in ([c] if c == G else [c, G]):
            op = op @ lat.SX(v, cc)
    return op

In [ ]:
# ---------------------------------------------------------- Knill-Laflamme distance lower bound
def _milp_binary(c_obj, A_eq_mod2, rhs_mod2, extra_ub=None, fixed_zero=()):
    """min c.x over binary x subject to A x = rhs (mod 2) (rows of A_eq_mod2), and optional
    linear constraints extra_ub = [(row, ub), ...] (row.x <= ub).
    Returns (value, x) if an optimum was found, (None, None) only if the problem is proven
    infeasible (scipy status 2).  Any other status (time/iteration limit, unbounded, numerical
    failure) raises RuntimeError, so that a solver failure is never mistaken for "no solution"."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    A = np.array(A_eq_mod2, dtype=float)
    m, n = A.shape
    nv = n + m
    c = np.concatenate([np.asarray(c_obj, float), np.zeros(m)])
    cons = [LinearConstraint(np.concatenate([A, -2 * np.eye(m)], axis=1), rhs_mod2, rhs_mod2)]
    for row, ub in (extra_ub or []):
        cons.append(LinearConstraint(np.concatenate([np.asarray(row, float), np.zeros(m)])[None, :], -np.inf, ub))
    ub_ = np.concatenate([np.ones(n), np.full(m, np.inf)])
    for i in fixed_zero:
        ub_[i] = 0
    res = milp(c, constraints=cons, integrality=np.ones(nv), bounds=Bounds(np.zeros(nv), ub_))
    if res.status == 2:
        return None, None
    if res.status != 0:
        raise RuntimeError(f'MILP solver did not finish (status {res.status}: {res.message})')
    return round(res.fun), np.round(res.x[:n]).astype(np.uint8)


def low_weight_stabilizer_shifts(cd, wmax, frozen=()):
    """Exact enumeration (MILP with no-good cuts) of all t in span(S) with 1 <= |t| <= wmax,
    where span(S) = C_Z  intersect  z_log^perp  (k = 1).  Frozen qubits are excluded
    (an X on a frozen qubit violates its single-qubit Z check, so t would leave C_Z).
    Returns (found, stop) where stop records why the enumeration ended:
    'infeasible' (no further t with 1 <= |t| <= wmax) or 'min weight > wmax'."""
    cols = [c for c in cd.cols if c not in set(frozen)]
    H = np.vstack([cd.HZ[:, cols], cd.z_log[cols][None, :]])
    H = H[np.any(H, axis=1)]
    rhs = np.zeros(len(H))
    n = len(cols)
    cuts = [(np.ones(n), wmax), (-np.ones(n), -1)]
    found = []
    while True:
        val, x = _milp_binary(np.ones(n), H, rhs, extra_ub=cuts)
        if val is None:
            stop = 'infeasible'
            break
        if val > wmax:
            stop = 'min weight > wmax'
            break
        sup = np.nonzero(x)[0]
        t = np.zeros(cd.lat.n, dtype=np.uint8)
        t[[cols[i] for i in sup]] = 1
        found.append(t)
        row = -np.ones(n)
        row[sup] = 1
        cuts.append((row, len(sup) - 1))                  # exclude exactly this vector
    return found, stop


def kl_distance_lower_bound(cd, wmax, frozen=()):
    r"""Rigorous lower-bound argument for the Knill-Laflamme distance of the (non-Pauli) code cd.

    For a Pauli E = X^t Z^u (up to phase):
      * t not in C_Z           -> E maps the code space to an orthogonal space: detectable;
      * t in the logical coset -> |t| >= X-shift distance (checked separately);
      * t in span(S)           -> X^t = S_c F_c with S_c a stabilizer and F_c = (-1)^{f_c} diagonal,
        so P E P = P F_c Z^u P.  Expanding (-1)^{f_c} in Z-type Paulis Z^v with v supported on
        Q_f = supp(f_c), a violation of KL requires some v + u to be a logical Z operator l.
        Hence |E| >= |t| + |u \ t| >= |t| + min_l |l \ (Q_f u t)|  =: LB_t.
    Returns (min over enumerated t of LB_t, ts, details, stop) where stop is the termination
    reason of the enumeration (see low_weight_stabilizer_shifts).  t = 0 is covered by d_Z."""
    lat = cd.lat
    ts, stop = low_weight_stabilizer_shifts(cd, wmax, frozen)
    cols = [c for c in cd.cols if c not in set(frozen)]
    Srows = cd.S[:, cols]
    Srows = Srows[np.any(Srows, axis=1)]
    best, details = None, []
    for t in ts:
        coef = f2_solve(cd.S, t)
        op = PSOp()
        for i in np.nonzero(coef)[0]:
            op = op @ cd.xgens[i][1]
        assert op.s == frozenset(np.nonzero(t)[0])
        Qf = {q for pr in op.quad for q in pr} | set(op.lin) | set(np.nonzero(t)[0])
        cost = np.array([0.0 if c in Qf else 1.0 for c in cols])
        A = np.vstack([Srows, cd.t_log[cols][None, :]])
        rhs = np.concatenate([np.zeros(len(Srows)), [1.0]])
        val, _ = _milp_binary(cost, A, rhs)
        assert val is not None                            # a logical Z always exists
        lb = int(t.sum()) + val
        details.append(dict(weight_t=int(t.sum()), Qf_size=len(Qf), LB=lb))
        best = lb if best is None else min(best, lb)
    return best, ts, details, stop

In [ ]:
# ---------------------------------------------------------- full resource tables (used by main + comparison)
def d4_resource_tables(D_LIST, dur=None, params=None, verbose=True):
    """Per-stage and summary tables of the Kobayashi protocol for every d in D_LIST, gate set
    ('primitive' | 'clifford_t'), counting variant (data_only), time model and comparison A/B.
    Returns (stage_df, sum_df, blocks)."""
    import pandas as pd, time
    dur = dur or DUR_UNIT
    params = params or DEFAULT_PARAMS
    records, summaries, blocks = [], [], {}
    for d in D_LIST:
        t0 = time.time()
        for gs in ['primitive', 'clifford_t']:
            cd = D4CostData(d, dur, decomposed=(gs == 'clifford_t'))
            for data_only in [False, True]:
                for model in ['FT_ref', 'NL_min']:
                    for comp in ['A', 'B']:
                        st = d4_stages(cd, model, params, comp, data_only=data_only)
                        for r in st:
                            r.update(gateset=gs, data_only=data_only)
                        records += st
                        s = summarize(st)
                        s.update(protocol='Kobayashi_D4', d=d, model=model, comparison=comp, gateset=gs,
                                 data_only=data_only)
                        summaries.append(s)
            blocks[f'{d}|{gs}'] = dict(d=d, gateset=gs, n=cd.n, n_total=cd.n_total, n_anc_g=cd.n_anc_g,
                                       round={k: {kk: (dict(vv) if isinstance(vv, Counter) else vv) for kk, vv in v.items()}
                                              for k, v in cd.round.items()},
                                       U_time=cd.U['time'], U_gates=dict(cd.U['gates']), out_time=cd.out['time'],
                                       out_gates=dict(cd.out['gates_worst']), xcorr_time=cd.x_corr['time'])
        if verbose:
            print(f'Kobayashi d={d}: {time.time() - t0:.1f}s')
    return jsonify_columns(pd.DataFrame(records)), jsonify_columns(pd.DataFrame(summaries)), blocks

### 雑音付き D4 code switching ライブラリ（回路・雑音モデル・Takada らの Pauli proxy 法による非 Clifford シミュレーション・復号器）

In [ ]:
"""noisy_d4_lib: circuit-level noisy simulation of the Kobayashi-Zhu-Hsin D4 code switching protocol.

Non-Clifford simulation method: Clifford-stabilizer simulation with Pauli proxies
(Y. Takada, S. D. Bartlett, D. J. Williamson, arXiv:2609.16929, Algorithm 2 (Sec. VII D) and Sec. VIII).
For every Monte Carlo shot the noisy non-Clifford circuit is mapped to a Clifford circuit (the
"Pauli-proxy circuit") with the same measurement statistics:
  * every diagonal third-level gate D (CCZ of the D4 gadgets, CS / CS^dag / T^dag of U) is removed;
  * the X component e of the sampled Pauli errors and of applied X corrections, propagated forward
    through the Clifford gates of the true circuit, is tracked ("continuing errors"); where e meets
    D the diagonal Clifford gate C_e = D^dag X^e D X^e (a product of S, Z, CZ) is inserted
    (= the controlled Pauli dressing Lambda(D_U(E)) of Takada Fig. 14 / Eq. (165));
  * Z components commute with every diagonal gate and need no propagation.
All Clifford gates, sampled Pauli errors and inserted dressings are executed by stim's TableauSimulator.

Requires (as globals, embedded before this code): numpy as np, stim, common_lib, d4_lib.
"""
import itertools
import numpy as np

In [ ]:
# ------------------------------------------------------------------ gate algebra
DIAG_THETA = {                     # phase polynomials (units of pi/4) of the diagonal 3rd-level gates
    'CCZ': lambda x: 4 * x[0] * x[1] * x[2],
    'NCCZ': lambda x: 4 * x[0] * x[1] * x[2],      # virtual noiseless CCZ (deferred switch-out correction)
    'CS': lambda x: 2 * x[0] * x[1],
    'CSDG': lambda x: 6 * x[0] * x[1],
    'T': lambda x: x[0],
    'TDG': lambda x: 7 * x[0],
}
NONCLIFF = set(DIAG_THETA)
_DRESS = {}


def diag_clifford_gates(delta, k):
    """delta: dict bits -> phase exponent mod 8 (even).  Returns gates ('S'|'Z'|'SDG', i), ('CZ', i, j)
    (local indices) equal to diag(omega^delta) up to a global phase; asserts the result is exact."""
    c0 = delta[(0,) * k]
    a, gates = [], []
    for i in range(k):
        e = tuple(int(j == i) for j in range(k))
        ai = (delta[e] - c0) % 8
        assert ai % 2 == 0, 'dressing is not Clifford'
        a.append(ai // 2)
        if ai // 2:
            gates.append(({1: 'S', 2: 'Z', 3: 'SDG'}[ai // 2], i))
    b = {}
    for i, j in itertools.combinations(range(k), 2):
        e = tuple(int(m in (i, j)) for m in range(k))
        v = (delta[e] - c0 - 2 * a[i] - 2 * a[j]) % 8
        assert v in (0, 4), 'dressing has a non-Clifford quadratic term'
        b[(i, j)] = v // 4
        if v:
            gates.append(('CZ', i, j))
    for x in itertools.product((0, 1), repeat=k):
        pred = c0 + sum(2 * a[i] * x[i] for i in range(k)) + sum(4 * b[ij] * x[ij[0]] * x[ij[1]] for ij in b)
        assert (pred - delta[x]) % 8 == 0, 'dressing has a cubic term'
    return gates


def dressing(name, e):
    """C_e = D^dag X^e D X^e for diagonal gate `name` and X pattern e (tuple of bits), local indices."""
    key = (name, e)
    if key not in _DRESS:
        th = DIAG_THETA[name]
        k = len(e)
        delta = {x: (th(tuple(xi ^ ei for xi, ei in zip(x, e))) - th(x)) % 8
                 for x in itertools.product((0, 1), repeat=k)}
        _DRESS[key] = diag_clifford_gates(delta, k)
    return _DRESS[key]

In [ ]:
# ------------------------------------------------------------------ noise model
class NoiseModel:
    """Standard circuit-level Pauli noise (Takada et al. App. A 2) plus optional idling noise:
    after every k-qubit unitary a k-qubit depolarizing channel (probability p; p_ccz for CCZ, p_nc for
    the gates of U), preparation followed by a flip (p_reset), measurement preceded by a flip (p_meas),
    every idle qubit of a time step depolarized with p_idle."""

    def __init__(self, p, p_meas=None, p_reset=None, p_idle=None, p_ccz=None, p_nc=None):
        self.p = p
        self.p_meas = p if p_meas is None else p_meas
        self.p_reset = p if p_reset is None else p_reset
        self.p_idle = p if p_idle is None else p_idle
        self.p_ccz = p if p_ccz is None else p_ccz
        self.p_nc = p if p_nc is None else p_nc

    def locations(self, ops, idle):
        """Noise locations of one time step: list of (when, kind, qubits, prob), when in {'pre','post'}."""
        loc = []
        for op in ops:
            nm, qs = op[0], op[1:]
            if nm in ('NCCZ', 'MZ0'):
                continue
            if nm == 'FLIPX':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MZ':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MX':
                loc.append(('pre', 'Z', qs, self.p_meas))
            elif nm == 'RZ':
                loc.append(('post', 'X', qs, self.p_reset))
            elif nm == 'RX':
                loc.append(('post', 'Z', qs, self.p_reset))
            elif nm == 'CCZ':
                loc.append(('post', 'D3', qs, self.p_ccz))
            elif nm in ('CS', 'CSDG', 'T', 'TDG'):
                loc.append(('post', 'D%d' % len(qs), qs, self.p_nc))
            else:
                loc.append(('post', 'D%d' % len(qs), qs, self.p))
        for q in idle:
            loc.append(('post', 'D1', (q,), self.p_idle))
        return [l for l in loc if l[3] > 0]


PAULI_CHOICES = {k: [''.join(p) for p in itertools.product('IXYZ', repeat=k) if set(p) != {'I'}]
                 for k in (1, 2, 3)}

In [ ]:
# ------------------------------------------------------------------ the noisy protocol circuit
class D4SwitchingCircuit:
    """Time-step level description of the noisy KZH protocol on the bounded square lattice.

    Blocks (in order):
      ('S0', r)  r = 1..R0   FSC |+_L> preparation: data |+>, rounds of all FSC checks
      ('S1', r)  r = 1..Rin  switch-in: g |0>, rounds of all D4 stabilizers ([KZH] step 2)
      ('U',)                 transversal U = W V ([KZH] eq. (8), step 3)
      ('SO',)                switch-out: Z^g readout ([KZH] step 4; deferred-correction form)
      ('S4', r)  r = 1..Rout FSC rounds after switching back ([KZH] step 4)
      ('FIN',)               one noiseless FSC round (fictitious error correction, Takada Sec. VI D)
    Qubits are re-indexed compactly: data (non-frozen r,g,b) first, then ancillas.
    sequential=True measures the gadgets of a commuting class one after another (validation only)."""

    def __init__(self, L, R0, Rin, Rout, sequential=False, n_pre_u=1):
        self.n_pre_u = n_pre_u
        P = D4Protocol(L)
        lat = P.lat
        self.P, self.lat, self.L = P, lat, L
        self.R0, self.Rin, self.Rout = R0, Rin, Rout
        data = sorted(P.data_qubits((R, G, B)))
        gadgets = {'D4': P.round_gadgets('D4'), 'FSC': P.round_gadgets('FSC')}
        ancs = sorted({g[0][1] for k in gadgets for g in gadgets[k]})
        c = {q: i for i, q in enumerate(data + ancs)}
        self.c = c
        self.nq, self.nd = len(data) + len(ancs), len(data)
        self.data_rb = sorted(c[q] for q in P.data_qubits((R, B)))
        self.data_g = sorted(c[q] for q in P.data_qubits((G,)))
        self.data_all = sorted(c[q] for q in data)
        conv = {'INIT_X': 'RX', 'INIT_Z': 'RZ', 'MEAS_X': 'MX', 'MEAS_Z': 'MZ', 'CNOT': 'CX'}
        cv = lambda g: (conv.get(g[0], g[0]),) + tuple(c[q] for q in g[1:])
        # label of every ancilla, per code
        self.lab_of = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            labs = [l for l, _ in cd.xgens] + [l for l in cd.zlabels if not P._is_frozen_check(l)]
            for l in labs:
                self.lab_of[(code, c[P.anc[l]])] = l

        def round_moments(code, which='all'):
            gl = gadgets[code]
            if which != 'all':
                gl = [g for g in gl if g[0][0] == ('INIT_Z' if which == 'Z' else 'INIT_X')]
            classes = commuting_classes(gl)
            if sequential:
                classes = [[g] for cls in classes for g in cls]
            moms = []
            for cls in classes:
                for layer in class_layers(cls):
                    moms.append([cv(g) for g in layer])
            return moms
        # D4 rounds: Pauli Z checks first, then (after the real-time flux correction) the X-type
        # Clifford stabilizers, so that fluxes detected by the Z checks are removed before they meet CCZs
        self.round_moms = {'FSC': round_moments('FSC'), 'D4z': round_moments('D4', 'Z'),
                           'D4x': round_moments('D4', 'X')}
        self.anc = {code: sorted({c[g[0][1]] for g in gadgets[code]}) for code in gadgets}
        self.U_moms = [[cv(g) for g in layer] for layer in class_layers([P.U_gates()])]
        # ---- switch-out data: S^g generators, Gamma (g configuration -> kappa), CZ^{r,b} pairs
        self.gens_g = [(lab, op) for lab, op in P.d4.xgens if lab[2] == 'g']
        gpos = {q: i for i, q in enumerate(self.data_g)}
        Sg = np.zeros((len(self.gens_g), len(self.data_g)), dtype=np.uint8)
        for i, (lab, op) in enumerate(self.gens_g):
            for q in op.s:
                if q in c and c[q] in gpos:
                    Sg[i, gpos[c[q]]] = 1
        self.Sg = Sg
        self.Gamma = left_inverse_T(Sg)
        self.gpairs = [[(c[i], c[j]) for (i, j) in op.quad if i in c and j in c] for lab, op in self.gens_g]
        self.network = [('NCCZ', self.data_g[e], i, j)
                        for v in range(len(self.gens_g)) for e in np.nonzero(self.Gamma[v])[0]
                        for (i, j) in self.gpairs[v]]
        # Pauli-frame strings Z^r Z^b for the S^g eigenvalues (protocol step 5 of the noiseless check V7)
        self.frame_strings = []
        for lab, op in self.gens_g:
            x0, y0 = lab[1]
            qs = [lat.q(lat.eidx[('v', x0, y)], cc) for y in range(y0) for cc in (R, B)]
            self.frame_strings.append(sorted(c[q] for q in qs if q in c))
        # ---- Z checks of the FSC (rb data, frozen checks removed) -> used for the S1 X frame
        self.zchecks = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            rows = []
            for lab, (l2, qs) in zip(cd.zlabels, lat.z_checks(cd.colours)):
                if P._is_frozen_check(lab):
                    continue
                rows.append((lab, sorted(c[q] for q in qs if q in c)))
            self.zchecks[code] = rows
        self.xgens = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            self.xgens[code] = [(lab, sorted(c[q] for q in op.s if q in c),
                                 [(c[i], c[j]) for (i, j) in op.quad if i in c and j in c]) for lab, op in cd.xgens]
        self._logical()
        self.blocks = self._blocks()

    # ------------------------------------------------------------ logical Clifford stabilizer
    def _logical(self):
        """Ideal output = D_F |+_L>_FSC with F(a) = w(a) + 4 sum_v c'_v(a) f^g_v(a) (protocol phase,
        validated in V7/V7b/V8).  Its logical Clifford stabilizer is H = D_F X^t D_F^dag = X^t D_Delta,
        Delta(x) = F(x+t) - F(x) = c0 + sum 2 a_i x_i + sum 4 b_ij x_i x_j (mod 8)."""
        P, lat, c = self.P, self.lat, self.c
        F = {}

        def add(mono, coef):
            m = frozenset(mono)
            if any(q not in c for q in m):
                return                                   # frozen qubits are |0>
            m = frozenset(c[q] for q in m)
            F[m] = (F.get(m, 0) + coef) % 8
        for coef, mono in W_operator(lat).terms:
            add(mono, coef)
        # c'_v(a): path sums along a BFS tree from the L_rb vertices (linear in a)
        from collections import deque
        root = lrb_vertices(lat)
        path = {v: [] for v in root}
        dq = deque(root)
        while dq:
            u = dq.popleft()
            for e in lat.incident(u):
                p, q = lat.endpoints(e)
                w = q if p == u else p
                if w not in path:
                    path[w] = path[u] + [e]
                    dq.append(w)
        for v in P.gverts:
            lin = [lat.q(e, cc) for e in path[v] for cc in (R, B)]
            for (i, j) in lat.cz_pairs(v, G):
                for k in lin:
                    add({k, i, j}, 4)
        self.Fpoly = {m: v for m, v in F.items() if v}
        t = sorted(c[int(q)] for q in np.nonzero(P.fsc.t_log)[0])
        self.t_log = t
        tset = set(t)
        D = {}
        for m, coef in self.Fpoly.items():           # F(x+t): substitute x_i -> 1 - x_i for i in t
            fixed = [q for q in m if q not in tset]
            flip = [q for q in m if q in tset]
            for sub in itertools.product((0, 1), repeat=len(flip)):
                mono = frozenset(fixed + [q for q, s in zip(flip, sub) if s])
                sign = (-1) ** sum(sub)
                D[mono] = (D.get(mono, 0) + coef * sign) % 8
        for m, coef in self.Fpoly.items():
            D[m] = (D.get(m, 0) - coef) % 8
        D = {m: v for m, v in D.items() if v}
        self.Delta_c0 = D.pop(frozenset(), 0)
        self.Delta_a, self.Delta_b = {}, {}
        for m, v in D.items():
            if len(m) == 1:
                assert v % 2 == 0, 'logical stabilizer not Clifford (odd linear term)'
                self.Delta_a[next(iter(m))] = v // 2 % 4
            elif len(m) == 2:
                assert v % 4 == 0, 'logical stabilizer not Clifford (quadratic term)'
                self.Delta_b[tuple(sorted(m))] = v // 4 % 2
            else:
                raise AssertionError('logical stabilizer not Clifford (cubic term)')
        self.Delta_a = {q: a for q, a in self.Delta_a.items() if a}
        self.Delta_b = {ij: b for ij, b in self.Delta_b.items() if b}

    def logical_proxy(self, fx, sign=1):
        """stim.PauliString of the Pauli proxy of the dressed logical stabilizer X^e H X^e
        (e = X frame fx on the data): proxy(X^t D_Delta R_e) = X^t * prod_{a_i=2} Z_i * R_e, where
        R_e = D_Delta^dag X^e D_Delta X^e = phase * Z-string (Takada Secs. VI D, VII B)."""
        n = self.nq
        ps = stim.PauliString(n)
        for q in self.t_log:
            ps[q] = 'X'
        zs = stim.PauliString(n)
        for q, a in self.Delta_a.items():
            if a == 2:
                zs[q] = 'Z'
        R = stim.PauliString(n)
        ph = 1
        for q, a in self.Delta_a.items():
            if fx[q] and a % 2 == 1:
                ph *= 1j ** a
                R *= _single(n, q, 'Z')
            elif fx[q] and a == 2:
                ph *= -1
        for (i, j), b in self.Delta_b.items():
            if fx[i] and fx[j]:
                ph *= -1
                R *= _single(n, i, 'Z') * _single(n, j, 'Z')
            elif fx[i]:
                R *= _single(n, j, 'Z')
            elif fx[j]:
                R *= _single(n, i, 'Z')
        out = ps * zs * R
        return out * complex(ph * sign)

    # ------------------------------------------------------------ blocks
    def _blocks(self):
        B_ = []
        rb, g, alld = self.data_rb, self.data_g, self.data_all
        for r in range(1, self.R0 + 1):
            pre = [[('RX', q) for q in rb]] if r == 1 else []
            B_.append(dict(name=('S0', r), code='FSC', moms=pre + self.round_moms['FSC'], data=rb))
        for r in range(1, self.Rin + 1 + self.n_pre_u):
            pre = [[('RZ', q) for q in g]] if r == 1 else []
            B_.append(dict(name=('S1', r, 'z'), code='D4', moms=pre + self.round_moms['D4z'], data=alld))
            if r <= self.Rin:            # rounds Rin+1.. have only the Z checks (flux checks before U)
                B_.append(dict(name=('S1', r, 'x'), code='D4', moms=self.round_moms['D4x'], data=alld))
        B_.append(dict(name=('U',), code=None, moms=self.U_moms, data=alld))
        B_.append(dict(name=('SO',), code=None,
                       moms=[[('FLIPX', q) for q in g], list(self.network), [('MZ0', q) for q in g]], data=alld))
        for r in range(1, self.Rout + 1):
            B_.append(dict(name=('S4', r), code='FSC', moms=self.round_moms['FSC'], data=rb))
        B_.append(dict(name=('FIN',), code='FSC', moms=self.round_moms['FSC'], data=rb, noiseless=True))
        return B_


def _single(n, q, p):
    s = stim.PauliString(n)
    s[q] = p
    return s


def left_inverse_T(S):
    """Gamma (m x n) over F2 with Gamma @ S^T = I_m, for S (m x n) of full row rank."""
    m, n = S.shape
    A = (S.T % 2).astype(np.uint8)
    M = np.concatenate([A, np.eye(n, dtype=np.uint8)], axis=1)
    r = 0
    for col in range(m):
        nz = np.nonzero(M[r:, col])[0]
        assert len(nz), 'S^T is not of full column rank'
        p = r + nz[0]
        M[[r, p]] = M[[p, r]]
        for rr in range(n):
            if rr != r and M[rr, col]:
                M[rr] ^= M[r]
        r += 1
    Gm = M[:m, m:]
    assert not ((Gm.astype(int) @ A.astype(int)) % 2 - np.eye(m, dtype=int)).any()
    return Gm

In [ ]:
# ------------------------------------------------------------------ exact reference (validation only)
class TrueSim:
    """Exact sparse state-vector simulation of the TRUE (non-Clifford) circuit, used to validate the
    Pauli-proxy simulation.  Basis states are uint64 bit masks: data qubits 0..nd-1, ancillas get a
    slot nd+k while they are alive (reset ... measurement)."""
    OMEGA = np.exp(1j * np.pi / 4)

    def __init__(self, nd, keys, amps):
        self.nd = nd
        self.keys = np.asarray(keys, dtype=np.uint64)
        self.amps = np.asarray(amps, dtype=complex)
        self.slot = {}
        self.free = []
        self.nslots = 0

    def bit(self, q):
        if q < self.nd:
            return q
        if q not in self.slot:
            raise KeyError(q)
        return self.slot[q]

    def _alloc(self, q):
        if q < self.nd:
            return q
        if q in self.slot:
            return self.slot[q]
        b = self.free.pop() if self.free else self.nd + self.nslots
        if b == self.nd + self.nslots:
            self.nslots += 1
        self.slot[q] = b
        assert b < 64
        return b

    def _bits(self, b):
        return ((self.keys >> np.uint64(b)) & np.uint64(1)).astype(np.int64)

    def _merge(self):
        u, inv = np.unique(self.keys, return_inverse=True)
        re = np.bincount(inv, weights=self.amps.real, minlength=len(u))
        im = np.bincount(inv, weights=self.amps.imag, minlength=len(u))
        a = re + 1j * im
        keep = np.abs(a) > 1e-12
        self.keys, self.amps = u[keep], a[keep]

    def X(self, q):
        self.keys = self.keys ^ np.uint64(1 << self.bit(q))

    def Z(self, q):
        self.amps = self.amps * (1 - 2 * self._bits(self.bit(q)))

    def Y(self, q):
        self.Z(q)
        self.X(q)
        self.amps = self.amps * 1j

    def H(self, q):
        b = self.bit(q)
        m = np.uint64(1 << b)
        v = self._bits(b)
        k0 = self.keys & ~m
        a = self.amps / np.sqrt(2)
        self.keys = np.concatenate([k0, k0 | m])
        self.amps = np.concatenate([a, a * (1 - 2 * v)])
        self._merge()

    def diag(self, qs, theta):
        bits = [self._bits(self.bit(q)) for q in qs]
        ph = np.array([theta(t) for t in zip(*bits)], dtype=float) if len(self.keys) else np.zeros(0)
        self.amps = self.amps * self.OMEGA ** ph

    def CX(self, c, t):
        vc = self._bits(self.bit(c)).astype(np.uint64)
        self.keys = self.keys ^ (vc << np.uint64(self.bit(t)))

    def measure_z(self, q, forced):
        b = self.bit(q)
        v = self._bits(b)
        p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
        prob = p1 if forced else 1 - p1
        keep = v == forced
        self.keys, self.amps = self.keys[keep], self.amps[keep]
        if prob > 1e-12:
            self.amps = self.amps / np.sqrt(prob)
        return prob

    def release(self, q, outcome):
        if q >= self.nd:
            if outcome:
                self.X(q)
            self.free.append(self.slot.pop(q))

    def reset(self, q, basis, rng):
        b = self._alloc(q)
        if q < self.nd:                      # data reset: measure and flip to |0>
            v = self._bits(b)
            p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
            o = int(rng.random() < p1)
            self.measure_z(q, o)
            if o:
                self.X(q)
        if basis == 'X':
            self.H(q)

    def run_moment(self, mom, errs, outcomes, rng):
        """Execute a time step with the given sampled errors; measurements forced to `outcomes`
        (list in the order of the moment's measurements).  Returns list of probabilities."""
        for when, ps, qs in errs:
            if when == 'pre':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        probs = []
        k = 0
        for op in mom:
            nm, qs = op[0], op[1:]
            if nm in DIAG_THETA:
                self.diag(qs, DIAG_THETA[nm])
            elif nm == 'CX':
                self.CX(*qs)
            elif nm == 'CZ':
                self.diag(qs, lambda x: 4 * x[0] * x[1])
            elif nm in ('S', 'SDG'):
                self.diag(qs, (lambda x: 2 * x[0]) if nm == 'S' else (lambda x: 6 * x[0]))
            elif nm in ('X', 'Z', 'Y'):
                getattr(self, nm)(qs[0])
            elif nm in ('RX', 'RZ'):
                self.reset(qs[0], nm[1], rng)
            elif nm in ('MX', 'MZ', 'MZ0'):
                q = qs[0]
                if nm == 'MX':
                    self.H(q)
                o = outcomes[k]
                k += 1
                probs.append(self.measure_z(q, o))
                if q >= self.nd:
                    self.release(q, o)
            elif nm == 'FLIPX':
                pass
            else:
                raise ValueError(nm)
        for when, ps, qs in errs:
            if when == 'post':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        return probs

    def expect_logical(self, circ):
        """<Psi| X^t D_Delta |Psi> on the data qubits (all ancillas released)."""
        assert not self.slot
        x = self.keys
        ph = np.full(len(x), circ.Delta_c0, dtype=float)
        bits = {}
        get = lambda q: bits.setdefault(q, ((x >> np.uint64(q)) & np.uint64(1)).astype(np.int64))
        for q, a in circ.Delta_a.items():
            ph += 2 * a * get(q)
        for (i, j), b in circ.Delta_b.items():
            ph += 4 * b * get(i) * get(j)
        Da = self.amps * self.OMEGA ** ph
        tm = np.uint64(sum(1 << q for q in circ.t_log))
        k2 = x ^ tm
        order = np.argsort(x)
        xs = x[order]
        pos = np.searchsorted(xs, k2)
        pos[pos >= len(xs)] = 0
        hit = xs[pos] == k2
        amp_t = np.zeros(len(x), dtype=complex)
        amp_t[hit] = self.amps[order][pos[hit]]
        return complex(np.sum(np.conj(amp_t) * Da))

In [ ]:
# ------------------------------------------------------------------ compiled protocol (fast Monte Carlo)
_KIND = {'X': 0, 'Z': 1, 'D1': 2, 'D2': 3, 'D3': 4}


class CompiledMoment:
    __slots__ = ('ops', 'base', 'meas', 'cx_c', 'cx_t', 'resets', 'clears', 'nonc', 'loc_slice')


class CompiledProtocol:
    """Pre-compiled time steps of a D4SwitchingCircuit + flattened noise-location table."""

    def __init__(self, circ, noise):
        self.circ, self.noise = circ, noise
        locs = []
        self.blocks = []
        live = set()
        for blk in circ.blocks:
            moms = []
            for mom in blk['moms']:
                cm = CompiledMoment()
                cm.ops = mom
                lines, meas, cxc, cxt, rs, cl, nc = [], [], [], [], [], [], []
                for op in mom:
                    nm, qs = op[0], op[1:]
                    if nm in NONCLIFF:
                        nc.append((nm, qs))
                    elif nm == 'FLIPX':
                        pass
                    elif nm == 'CX':
                        lines.append('CX %d %d' % qs)
                        cxc.append(qs[0]); cxt.append(qs[1])
                    elif nm == 'CZ':
                        lines.append('CZ %d %d' % qs)
                    elif nm in ('RX', 'RZ'):
                        lines.append(('RX %d' if nm == 'RX' else 'R %d') % qs[0])
                        rs.append(qs[0])
                    elif nm in ('MX', 'MZ', 'MZ0'):
                        lines.append(('MX %d' if nm == 'MX' else 'M %d') % qs[0])
                        meas.append((qs[0], 'X' if nm == 'MX' else 'Z'))
                        if nm == 'MZ0':
                            cl.append(qs[0])
                    else:
                        lines.append('%s %d' % (_STIM1[nm], qs[0]))
                cm.base = stim.Circuit('\n'.join(lines)) if lines else None
                cm.meas = meas
                cm.cx_c, cm.cx_t = np.array(cxc, dtype=np.int64), np.array(cxt, dtype=np.int64)
                cm.resets, cm.clears = np.array(rs, dtype=np.int64), np.array(cl, dtype=np.int64)
                cm.nonc = nc
                touched = {q for op in mom for q in op[1:]}
                idle = [q for q in blk['data'] if q not in touched] + [q for q in live if q not in touched]
                ml = [] if blk.get('noiseless') else noise.locations(mom, idle)
                s0 = len(locs)
                locs += ml
                cm.loc_slice = (s0, len(locs))
                for op in mom:
                    if op[0] in ('RX', 'RZ') and op[1] >= circ.nd:
                        live.add(op[1])
                    if op[0] in ('MX', 'MZ'):
                        live.discard(op[1])
                moms.append(cm)
            self.blocks.append(dict(name=blk['name'], code=blk['code'], moms=moms))
        self.loc_when = np.array([0 if l[0] == 'pre' else 1 for l in locs], dtype=np.int8)
        self.loc_kind = np.array([_KIND[l[1]] for l in locs], dtype=np.int8)
        self.loc_p = np.array([l[3] for l in locs])
        self.loc_q = [l[2] for l in locs]
        self.n_loc = len(locs)


class FastShot:
    """One Monte Carlo shot of the Pauli-proxy circuit (same semantics as ProxyRun, compiled)."""
    _G1 = {'S': 's', 'SDG': 's_dag', 'Z': 'z'}

    def __init__(self, comp, rng, forced=None):
        self.comp, self.rng, self.circ = comp, rng, comp.circ
        self.sim = stim.TableauSimulator(seed=int(rng.integers(2 ** 62)))
        self.sim.set_num_qubits(comp.circ.nq)
        self.fx = np.zeros(comp.circ.nq, dtype=np.uint8)
        if forced is not None:                  # {location index: pauli string}
            self.errs = {i: (comp.loc_when[i], ps, comp.loc_q[i]) for i, ps in forced.items()}
            self.n_err = len(forced)
            self.n_dress = 0
            return
        hit = np.nonzero(rng.random(comp.n_loc) < comp.loc_p)[0]
        self.errs = {}
        for i in hit:
            k = comp.loc_kind[i]
            qs = comp.loc_q[i]
            if k == 0:
                ps = 'X'
            elif k == 1:
                ps = 'Z'
            else:
                ch = PAULI_CHOICES[len(qs)]
                ps = ch[rng.integers(len(ch))]
            self.errs[i] = (comp.loc_when[i], ps, qs)
        self.n_err = len(hit)
        self.n_dress = 0

    def _pauli(self, ps, qs):
        sim, fx = self.sim, self.fx
        for pc, q in zip(ps, qs):
            if pc == 'X':
                sim.x(q); fx[q] ^= 1
            elif pc == 'Y':
                sim.y(q); fx[q] ^= 1
            elif pc == 'Z':
                sim.z(q)

    def run_block(self, bi):
        """Run block bi; returns list of (qubit, basis, outcome) of its measurements."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        nmeas = 0
        meas = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs] if b > a and self.errs else []
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        if gt[0] == 'CZ':
                            sim.cz(qs[gt[1]], qs[gt[2]])
                        else:
                            getattr(sim, self._G1[gt[0]])(qs[gt[1]])
                        self.n_dress += 1
            if cm.base is not None:
                sim.do_circuit(cm.base)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            nmeas += len(cm.meas)
            meas += cm.meas
        if not nmeas:
            return []
        rec = sim.current_measurement_record()[-nmeas:]
        return [(q, bb, int(o)) for (q, bb), o in zip(meas, rec)]

    def apply_x(self, qubits):
        for q in qubits:
            self.sim.x(q)
            self.fx[q] ^= 1

    def init_frame(self, zrows, qubits):
        """X frame at the start of the first non-Clifford block (S1).  S0 is a Clifford circuit, so stim's
        state is the true state E|ideal>, |ideal> = FSC|+_L> (all checks +1); the X part of E is any
        solution x of H_Z x = s (s = current Z-check signs): solutions differ by X stabilizers / logical X
        of |ideal>, which act trivially on it, and Algorithm 2 is exact for any such representation."""
        s = []
        for lab, qs in zrows:
            ps = stim.PauliString(self.circ.nq)
            for q in qs:
                ps[q] = 'Z'
            v = self.sim.peek_observable_expectation(ps)
            assert v in (1, -1), 'Z check not deterministic at the end of S0'
            s.append(0 if v == 1 else 1)
        H = np.zeros((len(zrows), len(qubits)), dtype=np.uint8)
        pos = {q: i for i, q in enumerate(qubits)}
        for r, (lab, qs) in enumerate(zrows):
            for q in qs:
                H[r, pos[q]] = 1
        x = f2_solve(H.T, np.array(s, dtype=np.uint8))      # x H^T = s  <=>  H x = s
        assert x is not None
        self.fx[:] = 0
        for i in np.nonzero(x)[0]:
            self.fx[qubits[i]] = 1
        return np.array(s)

    _G = {'CX': 'cx', 'CZ': 'cz', 'S': 's', 'SDG': 's_dag', 'Z': 'z', 'X': 'x', 'Y': 'y'}

    def run_block_lockstep(self, bi, ts, rng):
        """Validation: same as run_block, but op by op, with the exact simulation `ts` (TrueSim) of the
        TRUE non-Clifford circuit run in lockstep with the same errors.  Before every measurement stim's
        peek value (+1/-1 deterministic, 0 random) is compared with the exact probability of the outcome.
        Returns list of (qubit, basis, outcome, peek, exact probability)."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        out = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs]
            terrs = [('pre' if w == 0 else 'post', ps, qs) for w, ps, qs in errs]
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        getattr(sim, self._G[gt[0]])(*[qs[i] for i in gt[1:]])
                        self.n_dress += 1
            outs, peeks = [], []
            for op in cm.ops:
                nm, qs = op[0], op[1:]
                if nm in NONCLIFF or nm == 'FLIPX':
                    continue
                if nm in ('RX', 'RZ'):
                    (sim.reset_x if nm == 'RX' else sim.reset_z)(qs[0])
                elif nm in ('MX', 'MZ', 'MZ0'):
                    q = qs[0]
                    if nm == 'MX':
                        peeks.append(sim.peek_x(q)); sim.h(q); o = sim.measure(q); sim.h(q)
                    else:
                        peeks.append(sim.peek_z(q)); o = sim.measure(q)
                    outs.append(int(o))
                else:
                    getattr(sim, self._G[nm])(*qs)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            probs = ts.run_moment(cm.ops, terrs, outs, rng)
            out += [(q, bb, o, pk, pr) for (q, bb), o, pk, pr in zip(cm.meas, outs, peeks, probs)]
        return out

In [ ]:
# ------------------------------------------------------------------ decoders
def _wt(p):
    p = min(max(p, 1e-12), 0.5 - 1e-9)
    return float(np.log((1 - p) / p))


_XFRAC = {0: 1.0, 1: 0.0, 2: 2 / 3, 3: 8 / 15, 4: 32 / 63}       # prob. that a location puts X/Y on a qubit


class FluxDecoder:
    """Real-time ("just-in-time") decoding of the Pauli Z checks (non-Abelian fluxes = X errors).

    [KZH] step 2 pairs the fluxes "in real time ... whenever the observation time of the syndromes is
    comparable to the separation of the syndromes" with the decoder of [Dav] (not accessible here).
    Implementation used in this study (own construction, documented as such): after every round t the
    space-time matching graph of all rounds <= t (with an open time boundary after round t) is decoded
    by minimum-weight perfect matching (PyMatching); the space-like edges whose round is <= t - delay
    are committed and the corresponding X corrections are applied physically at once (feed-forward).
    Before the first non-Clifford block (end of S0) and before U (end of S1) everything is committed.
    The g readout of the switch-out is decoded together with the g fluxes of S1 and gives the
    corrected configuration used for kappa."""

    def __init__(self, comp, delay=0, gauge_w=0.05):
        circ = comp.circ
        self.comp, self.circ, self.delay, self.gauge_w = comp, circ, delay, gauge_w
        zr = {code: circ.zchecks[code] for code in ('D4', 'FSC')}
        g_rows = [(lab, qs) for lab, qs in zr['D4'] if lab[0] == 'plaq' and lab[2] == 'g']
        # flux rounds
        self.fr = []
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                self.fr.append(dict(block=bi, name=nm, rows=zr[blk['code']]))
            elif nm[0] == 'SO':
                self.fr.append(dict(block=bi, name=nm, rows=g_rows))
        off = 0
        for k, f in enumerate(self.fr):
            f['off'] = off
            f['idx'] = {lab: off + i for i, (lab, qs) in enumerate(f['rows'])}
            off += len(f['rows'])
        self.n_nodes = off
        # relation for the SO detectors: prod_{q in g-plaq} Z^g Z^r Z^b (pre-V) = sum of D4 rows
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        d4rows = zr['D4']
        H = np.zeros((len(d4rows), circ.nq), dtype=np.uint8)
        for i, (lab, qs) in enumerate(d4rows):
            H[i, qs] = 1
        self.so_rel = {}
        for lab, qs in g_rows:
            v = np.zeros(circ.nq, dtype=np.uint8)
            for q in qs:
                e = inv[q] // 3
                for cc in (R, G, B):
                    raw = lat.q(e, cc)
                    if raw in c:
                        v[c[raw]] ^= 1
            sol = f2_solve(H, v)
            assert sol is not None, 'SO relation not in the D4 Z-check span'
            self.so_rel[lab] = [d4rows[i][0] for i in np.nonzero(sol)[0]]
        # rows containing each qubit, per code
        self.rows_of = {code: {} for code in ('D4', 'FSC')}
        for code in ('D4', 'FSC'):
            for lab, qs in zr[code]:
                for q in qs:
                    self.rows_of[code].setdefault(q, []).append(lab)
        self._error_rates()
        self._build()

    def _error_rates(self):
        """Phenomenological edge probabilities estimated from the circuit-level noise locations."""
        comp = self.comp
        self.p_data = []           # per flux round: dict qubit -> prob of X/Y since the previous flux round
        self.p_meas = []           # per flux round: dict label -> prob that the check outcome is flipped
        prev_block = -1
        for f in self.fr:
            blk = comp.blocks[f['block']]
            pd, pa = {}, {}
            for bj in range(prev_block + 1, f['block'] + 1):
                for cm in comp.blocks[bj]['moms']:
                    a, b = cm.loc_slice
                    for i in range(a, b):
                        fr_ = _XFRAC[comp.loc_kind[i]] * comp.loc_p[i]
                        for q in comp.loc_q[i]:
                            if q < self.circ.nd:
                                pd[q] = pd.get(q, 0) + fr_
                            elif bj == f['block']:
                                pa[q] = pa.get(q, 0) + fr_
            prev_block = f['block']
            pm = {}
            if blk['code']:
                for lab, qs in f['rows']:
                    a = self.circ.P.anc[lab]
                    pm[lab] = pa.get(self.circ.c[a], 0)
            self.p_data.append(pd)
            self.p_meas.append(pm)

    def _build(self):
        """Matching graphs M_t for t = 0..len(fr)-1 (rounds <= t, open boundary after t)."""
        self.mt = []
        nfr = len(self.fr)
        for t in range(nfr):
            m = pymatching.Matching()
            look = {}

            def add(u, v, w, info):
                key = (u, v) if v == -1 else (min(u, v), max(u, v))
                if key in look and look[key][0] <= w:
                    return
                look[key] = (w, info)
            for k in range(t + 1):
                f = self.fr[k]
                code = self.comp.blocks[f['block']]['code'] or 'D4'
                rows = f['rows']
                rowset = {lab for lab, _ in rows}
                # space edges: X on a data qubit before round k
                qrows = {}
                for lab, qs in rows:
                    for q in qs:
                        qrows.setdefault(q, []).append(f['idx'][lab])
                for q, nodes in qrows.items():
                    if k == 0:
                        w = self.gauge_w           # random first-round outcomes of S0 (gauge choice)
                    else:
                        w = _wt(self.p_data[k].get(q, 1e-6))
                    if len(nodes) == 2:
                        add(nodes[0], nodes[1], w, ('s', q, k))
                    elif len(nodes) == 1:
                        add(nodes[0], -1, w, ('s', q, k))
                # time edges to the previous occurrence of the same label
                if k > 0:
                    for lab, qs in rows:
                        prev = self._prev(k, lab)
                        if prev is not None:
                            kp, node_p = prev
                            w = _wt(self.p_meas[kp].get(lab, 1e-6))
                            add(node_p, f['idx'][lab], w, ('t', lab, kp))
            # switch-out round: a measurement error of a check of the last D4 Z round enters the reference
            # of the g-readout detectors (hyper-edge with its next detector); represent it by an edge between
            # the two nodes known at this time and fix the stored reference when it is committed ('m')
            if self.fr[t]['name'][0] == 'SO':
                f = self.fr[t]
                kp = t - 1
                for lab, qs in f['rows']:
                    for l2 in self.so_rel[lab]:
                        if l2 != lab and l2 in self.fr[kp]['idx']:
                            add(self.fr[kp]['idx'][l2], f['idx'][lab], _wt(self.p_meas[kp].get(l2, 1e-6)),
                                ('m', l2, kp))
            # open time boundary: nodes whose label is measured again after t
            for k in range(t + 1):
                f = self.fr[k]
                for lab, qs in f['rows']:
                    if self._next(k, lab) is not None and self._next(k, lab) > t and self._latest(k, lab, t):
                        add(f['idx'][lab], -1, _wt(self.p_meas[k].get(lab, 1e-6) + 1e-9), ('f', lab, k))
            for (u, v), (w, info) in look.items():
                if v == -1:
                    m.add_boundary_edge(u, weight=w)
                else:
                    m.add_edge(u, v, weight=w)
            nn = self.fr[t]['off'] + len(self.fr[t]['rows'])
            self.mt.append((m, {k_: v[1] for k_, v in look.items()}, nn))

    def _prev(self, k, lab):
        for kp in range(k - 1, -1, -1):
            if lab in self.fr[kp]['idx']:
                return kp, self.fr[kp]['idx'][lab]
        return None

    def _next(self, k, lab):
        for kn in range(k + 1, len(self.fr)):
            if lab in self.fr[kn]['idx']:
                return kn
        return None

    def _latest(self, k, lab, t):
        """True if round k holds the latest occurrence of lab among rounds <= t."""
        for kk in range(k + 1, t + 1):
            if lab in self.fr[kk]['idx']:
                return False
        return True

    # ---------------------------------------------------------------- per shot
    def start(self):
        self.last = {}
        self.active = set()
        self.k = -1

    def push_round(self, k, outcomes_by_label=None, readout=None):
        """Add the detection events of flux round k.  outcomes_by_label: dict label -> bit (Z checks);
        readout: dict g-qubit -> bit (switch-out)."""
        f = self.fr[k]
        self.k = k
        if f['name'][0] == 'SO':
            par = {}
            for lab, qs in f['rows']:
                v = 0
                for q in qs:
                    v ^= readout[q]
                ref = 0
                for l2 in self.so_rel[lab]:
                    ref ^= self.last.get(l2, 0)
                par[lab] = v
                if v ^ ref:
                    self.active ^= {f['idx'][lab]}
            self.readout = dict(readout)
            return
        for lab, qs in f['rows']:
            m = outcomes_by_label[lab]
            if m ^ self.last.get(lab, 0):
                self.active ^= {f['idx'][lab]}
            self.last[lab] = m

    def decode(self, flush=False, min_age=None):
        """Decode with all rounds <= current; returns list of committed space corrections (qubit, k).
        Commits space-like edges with round k <= t - delay (flush: k <= t - min_age, default all)."""
        t = self.k
        m, look, nn = self.mt[t]
        if not self.active:
            return []
        syn = np.zeros(m.num_detectors, dtype=np.uint8)
        for u in self.active:
            syn[u] = 1
        edges = m.decode_to_edges_array(syn)
        out = []
        for u, v in edges:
            u, v = int(u), int(v)
            key = (u, -1) if v == -1 else ((v, -1) if u == -1 else (min(u, v), max(u, v)))
            info = look[key]
            if info[0] == 'm' and flush:
                self.active ^= {key[0], key[1]}
                self.last[info[1]] = self.last.get(info[1], 0) ^ 1     # it was a measurement error
                continue
            if info[0] != 's':
                continue
            q, k = info[1], info[2]
            lim = (t - (min_age or 0)) if flush else (t - self.delay)
            if k > lim:
                continue
            out.append((q, k))
            if key[1] == -1:
                self.active ^= {key[0]}
            else:
                self.active ^= {key[0], key[1]}
        return out

    def register_correction(self, q, code):
        """A physical X correction on data qubit q: flip the stored reference of every Z check of the
        current code containing q (as if it had been applied before that check was measured)."""
        for lab in self.rows_of[code].get(q, []):
            if lab in self.last:
                self.last[lab] ^= 1


class ChargeDecoder:
    """Decoding of the X-type checks (Abelian charges = Z errors) after the protocol: minimum-weight
    matching on the detector error model of the Pauli-proxy circuit (stim + PyMatching).  The DEM
    describes the Pauli errors of the circuit; the extra randomness produced by non-Abelian dressings
    is not in the model (it is present in the simulated data).  Output: predicted flip of the logical
    observable (proxy of the logical Clifford stabilizer, including the Pauli-frame strings)."""

    def __init__(self, comp, dem_noise=None, aligned=True):
        circ = comp.circ
        self.comp, self.circ = comp, circ
        nz = dem_noise or comp.noise
        # measurement bookkeeping
        meas_lab = []          # per absolute measurement: (block_name, kind, label or qubit)
        blk_meas = []
        for bi, blk in enumerate(comp.blocks):
            ids = []
            for cm in blk['moms']:
                for q, b in cm.meas:
                    if blk['code']:
                        lab = circ.lab_of[(blk['code'], q)]
                        meas_lab.append((blk['name'], b, lab))
                    else:
                        meas_lab.append((blk['name'], 'R', q))
                    ids.append(len(meas_lab) - 1)
            blk_meas.append(ids)
        self.n_meas = len(meas_lab)
        self.blk_meas = blk_meas
        idx = {}
        for i, (bn, b, lab) in enumerate(meas_lab):
            idx[(bn, lab)] = i
        self.meas_idx = idx
        sup = {code: {lab: set(xs) for lab, xs, quad in circ.xgens[code]} for code in ('D4', 'FSC')}
        names = [blk['name'] for blk in comp.blocks]
        S0 = [n for n in names if n[0] == 'S0']
        S1 = [n for n in names if n[0] == 'S1' and n[2] == 'x']
        S4 = [n for n in names if n[0] in ('S4', 'FIN')]
        dets = []
        fscl = [lab for lab, _, _ in circ.xgens['FSC']]
        d4l = [lab for lab, _, _ in circ.xgens['D4']]
        for j, n in enumerate(S0):
            for lab in fscl:
                dets.append([idx[(n, lab)]] + ([idx[(S0[j - 1], lab)]] if j else []))

        def span_solve(target, basis_labels, code):
            M = np.zeros((len(basis_labels), circ.nq), dtype=np.uint8)
            for i, l in enumerate(basis_labels):
                M[i, list(sup[code][l])] = 1
            v = np.zeros(circ.nq, dtype=np.uint8)
            v[list(target)] = 1
            sol = f2_solve(M, v)
            return None if sol is None else [basis_labels[i] for i in np.nonzero(sol)[0]]
        for j, n in enumerate(S1):
            for lab in d4l:
                if j == 0:
                    if not S0:
                        continue
                    combo = span_solve(sup['D4'][lab], fscl, 'FSC')
                    if combo is None:
                        continue                       # S^g: random first outcome (gauging)
                    dets.append([idx[(n, lab)]] + [idx[(S0[-1], l)] for l in combo])
                else:
                    dets.append([idx[(n, lab)], idx[(S1[j - 1], lab)]])
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        for j, n in enumerate(S4):
            first = {}
            for lab in fscl:
                if j == 0:
                    tgt = set(sup['FSC'][lab])
                    for q in list(sup['FSC'][lab]):
                        gq = lat.q(inv[q] // 3, G)
                        if gq in c:
                            tgt ^= {c[gq]}
                    combo = span_solve(tgt, d4l, 'D4')
                    assert combo is not None, 'S4 X check not determined by the last D4 round'
                    first[lab] = [idx[(n, lab)]] + [idx[(S1[-1], l)] for l in combo]
                else:
                    dets.append([idx[(n, lab)], idx[(S4[j - 1], lab)]])
            if j == 0:
                # detector basis across the switch-out: aligned=True uses, at a vertex with separate r and
                # b checks, D_r and D_r + D_b (S^g of the last D4 round cancels in D_r + D_b), so that a g
                # charge created during the last D4 round is a graph-like error, but an r charge then has
                # 4 symptoms; aligned=False (D_r, D_b) has the opposite property.  D4Experiment decodes
                # with both and keeps the lower-weight solution (no single basis makes all graph-like)
                for lab, d in first.items():
                    v, col = lab[1], lab[2]
                    if aligned and col == 'b' and ('X', v, 'r') in first:
                        dets.append(sorted(set(d) ^ set(first[('X', v, 'r')])))
                    else:
                        dets.append(d)
        self.dets = dets
        # logical observable: proxy logical (measured by MPP at the end of the DEM circuit) + frame terms
        tset = set(circ.t_log)
        self.cross = [len(set(fs) & tset) % 2 for fs in circ.frame_strings]
        self.obs_meas = [idx[(S1[-1], lab)] for (lab, op), cr in zip(circ.gens_g, self.cross) if cr]
        self._dem_circuit(nz)
        # sparse detector evaluation
        self.det_ptr = np.cumsum([0] + [len(d) for d in dets])
        self.det_idx = np.concatenate([np.array(d, dtype=np.int64) for d in dets])

    def _dem_circuit(self, nz):
        circ, comp = self.circ, self.comp
        lines = []
        for bi, blk in enumerate(comp.blocks):
            noiseless = blk['name'][0] == 'FIN'
            for cm in blk['moms']:
                pre, post = [], []
                touched = {q for op in cm.ops for q in op[1:]}
                a, b = cm.loc_slice
                for i in range(a, b):
                    k, p, qs = comp.loc_kind[i], comp.loc_p[i], comp.loc_q[i]
                    if k == 0:
                        ln = 'X_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 1:
                        ln = 'Z_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 2:
                        ln = 'DEPOLARIZE1(%.12g) %d' % (p, qs[0])
                    elif k == 3:
                        ln = 'DEPOLARIZE2(%.12g) %d %d' % (p, qs[0], qs[1])
                    else:
                        ln = 'DEPOLARIZE1(%.12g) %d %d %d' % (p * 48 / 63, *qs)
                    (pre if comp.loc_when[i] == 0 else post).append(ln)
                lines += pre
                if cm.base is not None:
                    lines.append(str(cm.base))
                lines += post
            lines.append('TICK')
        n = circ.nq
        lp = ['X%d' % q for q in circ.t_log if circ.Delta_a.get(q) != 2]
        lp += ['Y%d' % q for q in circ.t_log if circ.Delta_a.get(q) == 2]
        lp += ['Z%d' % q for q, a in circ.Delta_a.items() if a == 2 and q not in set(circ.t_log)]
        lines.append('MPP ' + '*'.join(lp))
        tot = self.n_meas + 1
        for d in self.dets:
            lines.append('DETECTOR ' + ' '.join('rec[%d]' % (i - tot) for i in d))
        lines.append('OBSERVABLE_INCLUDE(0) rec[-1] ' + ' '.join('rec[%d]' % (i - tot) for i in self.obs_meas))
        self.stim_circuit = stim.Circuit('\n'.join(lines))
        dem = self.stim_circuit.detector_error_model(decompose_errors=True, ignore_decomposition_failures=True,
                                                     approximate_disjoint_errors=True)
        self.matching = pymatching.Matching.from_detector_error_model(dem)
        # graph-like components of the DEM (for the flux-informed re-weighting)
        comp = {}
        for ins in dem.flattened():
            if ins.type != 'error':
                continue
            p = ins.args_copy()[0]
            cur_d, cur_o = [], 0
            parts = []
            for t in ins.targets_copy():
                if t.is_separator():
                    parts.append((tuple(sorted(cur_d)), cur_o)); cur_d, cur_o = [], 0
                elif t.is_relative_detector_id():
                    cur_d.append(t.val)
                elif t.is_logical_observable_id():
                    cur_o ^= 1
            parts.append((tuple(sorted(cur_d)), cur_o))
            for key in parts:
                if not key[0] or len(key[0]) > 2:
                    continue
                q0 = comp.get(key, 0.0)
                comp[key] = q0 * (1 - p) + p * (1 - q0)
        keys = list(comp)
        self.comp_keys = keys
        self.comp_index = {k[0]: i for i, k in enumerate(keys)}
        from scipy.sparse import csc_matrix
        rows, cols = [], []
        for j, (dd, o) in enumerate(keys):
            for dtt in dd:
                rows.append(dtt); cols.append(j)
        nd = self.stim_circuit.num_detectors
        self.H = csc_matrix((np.ones(len(rows), dtype=np.uint8), (rows, cols)), shape=(nd, len(keys)))
        self.O = csc_matrix((np.ones(sum(o for _, o in keys), dtype=np.uint8),
                             ([0] * sum(o for _, o in keys), [j for j, (dd, o) in enumerate(keys) if o])),
                            shape=(1, len(keys)))
        self.w0 = np.array([_wt(comp[k]) for k in keys])
        # detectors containing each measurement
        self.dets_of_meas = {}
        for j, d_ in enumerate(self.dets):
            for i in d_:
                self.dets_of_meas.setdefault(i, []).append(j)

    def detectors(self, outcomes):
        o = np.asarray(outcomes, dtype=np.uint8)[self.det_idx]
        return np.bitwise_xor.reduceat(o, self.det_ptr[:-1]) if len(o) else np.zeros(0, np.uint8)

    def flux_erasures(self, flux_events, w_erase=0.1):
        """Flux-informed re-weighting (our construction, following [KZH]'s description of cleaning up the
        Abelian charges after the fluxes have been paired): a flux on qubit q present during the X-type
        round ('S1', r, 'x') randomizes the outcomes of the D4 stabilizers whose CZ dressing contains q and
        leaves Z errors on the CZ partners with probability 1/2.  The corresponding time-like components
        and the space-like components of the partner Z errors get the weight w_erase."""
        circ = self.circ
        if not hasattr(self, '_cz_of'):
            self._cz_of = {}
            for lab, xs, quad in circ.xgens['D4']:
                for (i, j) in quad:
                    self._cz_of.setdefault(i, []).append((lab, j))
                    self._cz_of.setdefault(j, []).append((lab, i))
            self._stars = {}
            for lab, xs, quad in circ.xgens['D4']:
                for q in xs:
                    self._stars.setdefault(q, []).append(lab)
        er = set()
        for q, r in flux_events:
            blk = ('S1', r, 'x')
            nxt = ('S1', r + 1, 'x')
            for lab, partner in self._cz_of.get(q, []):
                m = self.meas_idx.get((blk, lab))
                if m is None:
                    continue
                dd = tuple(sorted(self.dets_of_meas.get(m, [])))
                if 1 <= len(dd) <= 2 and dd in self.comp_index:
                    er.add(self.comp_index[dd])
                # space-like component: Z on the partner flips the two stars containing it
                st = self._stars.get(partner, [])
                for rr in (blk, nxt):
                    ds = []
                    for l2 in st:
                        m2 = self.meas_idx.get((rr, l2))
                        if m2 is not None:
                            ds += [j for j in self.dets_of_meas.get(m2, []) if self.dets[j][0] == m2]
                    dd = tuple(sorted(ds))
                    if 1 <= len(dd) <= 2 and dd in self.comp_index:
                        er.add(self.comp_index[dd])
        return er

    def predict(self, outcomes, erasures=None, w_erase=0.1):
        """Returns (predicted flip of the logical observable, weight of the matching)."""
        det = self.detectors(outcomes)
        if not erasures:
            pred, w = self.matching.decode(det, return_weight=True)
            return int(pred[0]), float(w)
        w = self.w0.copy()
        w[list(erasures)] = w_erase
        m = pymatching.Matching.from_check_matrix(self.H, weights=w, faults_matrix=self.O)
        pred, wt = m.decode(det, return_weight=True)
        return int(pred[0]), float(wt)


class D4Experiment:
    """Noisy KZH protocol + decoders.  run(shots) returns per-shot logical infidelity samples
    (0, 1/2 or 1; Takada Sec. VI D with the infidelity (1 - <H>)/2 of the output) and diagnostics."""

    def __init__(self, L, R0, Rin, Rout, noise, delay=0, pre_u_age=0, n_pre_u=1, flux_informed=True,
                 quiet_window=False):
        self.pre_u_age = pre_u_age
        self.flux_informed = flux_informed
        self.circ = D4SwitchingCircuit(L, R0, Rin, Rout, n_pre_u=n_pre_u)
        self.comp = CompiledProtocol(self.circ, noise)
        self.noise = noise
        self.flux = FluxDecoder(self.comp, delay=delay)
        self.charges = [ChargeDecoder(self.comp, aligned=a) for a in (True, False)]
        # diagnostic (not a physical model): no noise in the switching window = the last D4 round, the Z
        # rounds before U, U, the switch-out and the first FSC round, where the single faults that our
        # decoders fail to correct are located (validation N5).  Gives an optimistic bound.  The decoders keep
        # the priors of the full noise model (they are built before the window is silenced).
        self.quiet_window = quiet_window
        if quiet_window:
            win = {('S1', Rin, 'z'), ('S1', Rin, 'x'), ('U',), ('SO',), ('S4', 1)}
            win |= {('S1', r, 'z') for r in range(Rin + 1, Rin + 1 + n_pre_u)}
            for blk in self.comp.blocks:
                if blk['name'] in win:
                    a, b = blk['moms'][0].loc_slice[0], blk['moms'][-1].loc_slice[1]
                    self.comp.loc_p[a:b] = 0.0
        self.gset = set(self.circ.data_g)

    def shot(self, rng, decode=True, forced=None):
        circ, comp, fl = self.circ, self.comp, self.flux
        sh = FastShot(comp, rng, forced)
        fl.start()
        outs = []
        kfr = 0
        m_g_last = None
        flux_events = []
        stats = dict(n_err=sh.n_err, n_corr=0)
        names = [b['name'] for b in comp.blocks]
        last_S0 = max([i for i, n in enumerate(names) if n[0] == 'S0'], default=-1)
        last_S1 = max(i for i, n in enumerate(names) if n[0] == 'S1')          # Z-only round before U
        last_S1x = max(i for i, n in enumerate(names) if n[0] == 'S1' and n[2] == 'x')
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm == ('S1', 1, 'z'):
                sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
            res = sh.run_block(bi)
            outs += [o for (_, _, o) in res]
            if bi == last_S1x:
                gl = {circ.lab_of[('D4', q)]: o for (q, b, o) in res if b == 'X'}
                m_g_last = [gl[lab] for lab, _ in circ.gens_g]
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                code = blk['code']
                zl = {circ.lab_of[(code, q)]: o for (q, b, o) in res if b == 'Z'}
                fl.push_round(kfr, zl)
                kfr += 1
                if decode:
                    flush = bi in (last_S0, last_S1) or nm[0] == 'FIN'
                    # before U: corrections supported only by the newest round stay undecided (decoded
                    # after the switch-out with the S4 / g-readout data)
                    cc = fl.decode(flush=flush, min_age=self.pre_u_age if bi == last_S1 else 0)
                    corr = [q for q, k in cc]
                    if nm[0] == 'S1':
                        # the flux was present during the X-type round preceding this Z round
                        flux_events += [(q, nm[1] - 1) for q in corr if nm[1] >= 2]
                    sh.apply_x(corr)
                    for q in corr:
                        fl.register_correction(q, code)
                    stats['n_corr'] += len(corr)
            elif nm[0] == 'SO':
                readout = {q: o for (q, b, o) in res}
                fl.push_round(kfr, readout=readout)
                kfr += 1
                a_dec = dict(readout)
                if decode:
                    phys = []
                    for q, k in fl.decode(flush=True):
                        if q in self.gset:
                            a_dec[q] ^= 1
                        else:
                            phys.append(q)
                    sh.apply_x(phys)
                    for q in phys:
                        fl.register_correction(q, 'FSC')
                    stats['n_corr'] += len(phys)
                self._switch_out_fix(sh, readout, a_dec, rng, noisy=forced is None and not self.quiet_window)
        # Pauli frame of the S^g eigenvalues (last S1 round), then the logical Clifford stabilizer
        for v, m in enumerate(m_g_last):
            if m:
                for q in circ.frame_strings[v]:
                    sh.sim.z(q)
        val = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
        if decode:
            sols = [ch.predict(outs, ch.flux_erasures(flux_events) if self.flux_informed else None)
                    for ch in self.charges]
            flip = min(sols, key=lambda s: s[1])[0]
        else:
            flip = 0
        val = val * (-1) ** flip
        stats['n_dress'] = sh.n_dress
        return (1 - val) / 2, stats

    def _switch_out_fix(self, sh, readout, a_dec, rng, noisy=True):
        """kappa = Gamma(a): the virtual network applied CZ(Gamma(a_raw)); the protocol applies
        CZ(Gamma(a_dec)) physically -> add the difference (Clifford) and the noise of the physical CZs."""
        circ = self.circ
        ar = np.array([readout[q] for q in circ.data_g], dtype=np.uint8)
        ad = np.array([a_dec[q] for q in circ.data_g], dtype=np.uint8)
        kr = (circ.Gamma.astype(int) @ ar) % 2
        kd = (circ.Gamma.astype(int) @ ad) % 2
        p = self.noise.p
        for v in range(len(circ.gens_g)):
            if kr[v] ^ kd[v]:
                for (i, j) in circ.gpairs[v]:
                    sh.sim.cz(i, j)
            if kd[v] and p > 0 and noisy:
                for (i, j) in circ.gpairs[v]:
                    if rng.random() < p:
                        ps = PAULI_CHOICES[2][rng.integers(15)]
                        sh._pauli(ps, (i, j))

    def run(self, shots, rng, decode=True):
        inf = np.zeros(shots)
        st = []
        for s in range(shots):
            inf[s], d = self.shot(rng, decode)
            st.append(d)
        return inf, st

In [ ]:
# ------------------------------------------------------------------ applicability of Takada et al.
def takada_conditions(L):
    """Algebraic checks of the sufficient conditions of Takada et al. (Definition 2 and Sec. VIII) for
    the D4 code switching protocol on the L x L lattice.  Returns dict name -> (passed, detail)."""
    P = D4Protocol(L)
    lat, d4, fsc = P.lat, P.d4, P.fsc
    res = {}
    # (a) admissible factorisation: every X-type Clifford stabilizer = X factors x CZ factors with disjoint
    #     supports (transversal, so every effectively measured operator stays Hermitian, Eq. (131))
    bad = 0
    for lab, op in d4.xgens:
        qs = list(op.s) + [q for pr in op.quad for q in pr]
        bad += len(qs) != len(set(qs))
    res['factors transversal (X and CZ factors on disjoint qubits)'] = (bad == 0, '%d generators' % len(d4.xgens))
    # (b) Pauli dressings: X/Y errors on the Clifford factors (CZ; S, CZ in the logical stabilizer; S,
    #     CZ dressings) give weight-1 Z dressings; all dressings commute (Eq. (91))
    I2 = np.eye(2); X = np.array([[0, 1], [1, 0]]); Y = np.array([[0, -1j], [1j, 0]]); Zm = np.diag([1, -1])
    CZm = np.diag([1, 1, 1, -1]); Sm = np.diag([1, 1j])
    ok = True
    for E in (X, Y):
        D = np.kron(E, I2) @ CZm @ np.kron(E, I2).conj().T @ CZm.conj().T       # [E_1, CZ]
        ok &= np.allclose(np.abs(D), np.abs(np.kron(I2, Zm)))
        D1 = E @ Sm @ E.conj().T @ Sm.conj().T
        ok &= np.allclose(np.abs(D1), np.abs(Zm))
    res['dressings of CZ / S by X, Y are single-qubit Z (dressing axis Z on every qubit)'] = (bool(ok), '')
    # (c) effective commutation: group commutators of measured generators are Z-type Paulis in the
    #     persistent Z-stabilizer span (D4 Z checks), Eq. (58) / Lemma 1
    HZr, HZp = f2_rref(d4.HZ)
    nbad, npair = 0, 0
    for (l1, a), (l2, b) in itertools.combinations(d4.xgens, 2):
        lin, const = commutator_affine(a, b)
        npair += 1
        if lin and not f2_in_rowspace(HZr, HZp, lat.vec(lin)):
            nbad += 1
    res['D4 generators: commutators in the Z-stabilizer span'] = (nbad == 0, '%d pairs' % npair)
    # switch-in: S^g measured on FSC|+_L> (x) |0>_g: commutators with the FSC generators and the logical X
    # must lie in the span of the FSC Z checks and the single-qubit Z^g (current Z stabilizers)
    # The state before switch-in, FSC|+_L> (x) |0>_g, is stabilized by the FSC Z checks, the single-qubit
    # Z^g, and by the D4-dressed X-type operators S^r_v, S^b_v (their CZ^{g,.} dressing acts trivially
    # on |0>_g) and a dressed logical X = X^t D (D diagonal Clifford, also trivial on |0>_g).  Takada
    # Sec. VIII B: the generators may be chosen such that the new measurements (S^g) effectively commute
    # or anticommute with them.  Check this choice:
    zin = [rows for lab, rows in lat.z_checks((R, B))] + [{lat.q(e, G)} for e in range(lat.E)]
    Hin = np.array([lat.vec(r) for r in zin], dtype=np.uint8)
    Hir, Hip = f2_rref(Hin)
    gens_in = [op for lab, op in d4.xgens if lab[2] != 'g']
    nbad = 0
    for lab, op in d4.xgens:
        for g in gens_in:
            lin, const = commutator_affine(op, g)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
    # dressed logical X: find D = prod CZ^{c_k} over the CZ pairs of the D4 generators with
    # [X^t D, S] in the D4 Z-stabilizer span for every X-type generator S (linear system over F2)
    t = set(int(q) for q in np.nonzero(fsc.t_log)[0])
    pairs = sorted({pr for _, op in d4.xgens for pr in op.quad})
    rows_eq = []                          # for each generator: commutator linear part as function of c
    HZr2, HZp2 = f2_rref(d4.HZ)
    # basis of the complement of the Z span to express "lin in span" as linear equations
    Npre = f2_nullspace(d4.HZ)            # vectors orthogonal to the span: lin in span <=> N.lin = 0
    A_cols, b_vec = [], None
    base_list = []
    for lab, op in d4.xgens:
        lin0, _ = commutator_affine(PSOp(s=t), op)
        base_list.append(lat.vec(lin0) if lin0 else np.zeros(lat.n, dtype=np.uint8))
    for pr in pairs:
        col = []
        for lab, op in d4.xgens:
            lin, _ = commutator_affine(PSOp(quad=[pr]), op)
            col.append(lat.vec(lin) if lin else np.zeros(lat.n, dtype=np.uint8))
        A_cols.append(col)
    # equations: N @ (base_v + sum_k c_k col_k,v) = 0 for all v
    eqA = np.array([[int((Npre.astype(int) @ A_cols[k][v].astype(int) % 2)[i]) for k in range(len(pairs))]
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    eqb = np.array([int((Npre.astype(int) @ base_list[v].astype(int) % 2)[i])
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    sol = f2_solve(eqA.T, eqb) if eqA.size else np.zeros(0, dtype=np.uint8)
    logical_ok = sol is not None
    if logical_ok:
        D = [pr for pr, cbit in zip(pairs, sol) if cbit]
        Xbar = PSOp(s=t, quad=D)
        # dressing trivial on |0>_g: every CZ of D touches a g qubit
        triv = all(any(q % 3 == G for q in pr) for pr in D)
        for lab, op in d4.xgens:
            lin, const = commutator_affine(Xbar, op)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
        logical_ok = triv
    res['switch-in: S^g vs dressed generators of FSC|+_L>|0>_g (current Z span), dressed logical X exists'] = (
        nbad == 0 and logical_ok, 'logical dressing with %s CZ pairs' % (len(D) if logical_ok else '-'))
    # (d) primary-support exactness (Definition 1): X supports of the X-type generators (and the logical
    #     X) are linearly independent, so no product of them is a non-Pauli operator without X part
    Ssup = np.vstack([d4.S, d4.t_log[None, :]])
    rk = f2_rank(Ssup[:, d4.cols])
    res['primary-support exactness (independent X supports)'] = (rk == Ssup.shape[0], 'rank %d / %d' % (rk, Ssup.shape[0]))
    # (e) the transversal U and the deferred switch-out network consist of CNOTs and diagonal third-level
    #     gates only (Sec. VIII A, Eq. (178)): Pauli / diagonal-Clifford errors stay of that form
    names = {g[0] for g in P.U_gates()}
    res['U = CNOT + diagonal 3rd-level gates (Sec. VIII A)'] = (names <= {'CNOT', 'CS', 'CSDG', 'TDG'}, str(sorted(names)))
    # (f) logical Clifford stabilizer of the output: Clifford (no cubic / odd terms) and effectively
    #     commuting with every FSC X check (dressing = Z string in the FSC Z span with sign +-1)
    circ = D4SwitchingCircuit(L, 1, 1, 1)
    Hf = np.array([[1 if q in qs else 0 for q in range(circ.nq)] for lab, qs in circ.zchecks['FSC']], dtype=np.uint8)
    Hfr, Hfp = f2_rref(Hf)
    nbad = 0
    for lab, xs, quad in circ.xgens['FSC']:
        fx = np.zeros(circ.nq, dtype=np.uint8)
        fx[xs] = 1
        R_ = circ.logical_proxy(fx) * circ.logical_proxy(np.zeros(circ.nq, dtype=np.uint8))
        zpart = np.array([1 if R_[q] in (2, 3) else 0 for q in range(circ.nq)], dtype=np.uint8)
        xpart = np.array([1 if R_[q] in (1, 2) else 0 for q in range(circ.nq)], dtype=np.uint8)
        if xpart.any() or (zpart.any() and not f2_in_rowspace(Hfr, Hfp, zpart)) or R_.sign.imag != 0:
            nbad += 1
    res['output logical stabilizer: Clifford, effectively commutes with FSC X checks'] = (nbad == 0, '')
    return res


def lockstep_validation(L, Rin, Rout, noise, rng, sequential=False, corr_prob=0.05):
    """Run the Pauli-proxy shot (FastShot) and the exact simulation of the TRUE circuit (TrueSim) in
    lockstep from the start of S1 (initial state E|+_L>_FSC|0>_g with a random Pauli E on the FSC data),
    with the same sampled circuit-level errors and random feed-forward X corrections.  Returns dict with
    the number of measurements, the number of mismatches (deterministic vs random, probability of the
    outcome), and the final <H> from both simulations."""
    circ = D4SwitchingCircuit(L, 0, Rin, Rout, sequential=sequential)
    comp = CompiledProtocol(circ, noise)
    P, c, fsc = circ.P, circ.c, circ.P.fsc
    zero = coset_state(fsc, set())
    one = coset_state(fsc, set(int(q) for q in np.nonzero(fsc.t_log)[0]))
    amps = {}
    for st in (zero, one):
        for k, v in st.a.items():
            amps[k] = amps.get(k, 0) + v / np.sqrt(2)
    keys = [sum(1 << c[q] for q in k) for k in amps]
    ts = TrueSim(circ.nd, keys, list(amps.values()))
    sh = FastShot(comp, rng)
    n = circ.nq
    stabs = []
    for lab, xs, quad in circ.xgens['FSC']:
        ps = stim.PauliString(n)
        for q in xs:
            ps[q] = 'X'
        stabs.append(ps)
    for lab, qs in circ.zchecks['FSC']:
        ps = stim.PauliString(n)
        for q in qs:
            ps[q] = 'Z'
        stabs.append(ps)
    ps = stim.PauliString(n)
    for q in circ.t_log:
        ps[q] = 'X'
    stabs.append(ps)
    for q in range(n):
        if q not in circ.data_rb:
            ps = stim.PauliString(n)
            ps[q] = 'Z'
            stabs.append(ps)
    sh.sim.set_state_from_stabilizers(stabs, allow_redundant=True)
    for q in rng.choice(circ.data_rb, 2, replace=False):
        pc = 'XYZ'[rng.integers(3)]
        getattr(ts, pc)(int(q)); getattr(sh.sim, pc.lower())(int(q))
    sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
    nmeas = nbad = nrand = 0
    m_g = None
    for bi, blk in enumerate(comp.blocks):
        res = sh.run_block_lockstep(bi, ts, rng)
        for q, bb, o, pk, pr in res:
            nmeas += 1
            if pk == 0:
                nrand += 1
                nbad += abs(pr - 0.5) > 1e-9
            else:
                nbad += (o != (0 if pk == 1 else 1)) or abs(pr - 1) > 1e-9
        nm = blk['name']
        if nm[0] == 'S1' and nm[2] == 'x':
            gl = {circ.lab_of[('D4', q)]: o for q, bb, o, pk, pr in res if bb == 'X'}
            m_g = [gl[lab] for lab, _ in circ.gens_g]
        if corr_prob and nm[0] in ('S1', 'S4') and rng.random() < corr_prob * len(blk['moms']):
            q = int(rng.choice(circ.data_rb if nm[0] == 'S4' else circ.data_all))
            sh.apply_x([q]); ts.X(q)
    for v, m in enumerate(m_g):
        if m:
            for q in circ.frame_strings[v]:
                sh.sim.z(q); ts.Z(q)
    true = ts.expect_logical(circ)
    proxy = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
    return dict(n_meas=nmeas, n_random=nrand, n_mismatch=int(nbad), H_true=true, H_proxy=proxy,
                n_dress=sh.n_dress, agree=bool(nbad == 0 and abs(true - proxy) < 1e-9))

### 雑音付き 15-to-1 ライブラリ（Litinski の誤り予算・密度行列計算・GoSC 配置・表面符号の p_L 較正）

In [ ]:
"""msd_noisy_lib: output error and cost of 15-to-1 magic-state distillation under circuit-level noise.

Error model (D. Litinski, "Magic State Distillation: Not as Costly as You Think", Quantum 3, 205 (2019),
and its published resource-estimation code github.com/litinski/magicstates, Apache-2.0; re-implemented
here, the error budgets are transcribed from that code):
  * the distillation circuit acts on 5 logical qubits (surface-code patches) and consists of Z-type
    pi/8 Pauli-product rotations; a faulty rotation is followed by a P_{pi/2}, P_{-pi/4} or P_{pi/4}
    error with probabilities p1, p2, p3 (faulty magic state: p/3 each; lattice-surgery measurement
    errors and errors in the ancilla region add to p1, p2);
  * patches suffer X / Z storage errors with probability 0.5 * p_L(d) per code cycle each;
  * p_L(p, d) = 0.1 (100 p)^((d+1)/2) per patch and code cycle (GoSC Eq. (10)), or a calibration of the
    same form obtained by stim simulations of the rotated surface code under the SAME circuit-level
    noise model as the D4 simulation (`calibrate_surface_code`);
  * the output error is the infidelity of the post-selected output state (exact density-matrix
    calculation, 128-bit arithmetic), the success probability is the post-selection probability.
Protocols:
  * `litinski2019_one_level(p, dx, dz, dm)` and `litinski2019_two_level(...)`: Litinski's optimised
    factories (transcribed; validated against the outputs stored in his notebook);
  * `gosc_15to1(p, d)` and `gosc_225to1(p, d1, d2)`: the A Game of Surface Codes Sec. 3 layouts
    (Fig. 17c: 11 tiles x 11 tau; Fig. 19: 176 tiles x 15 tau) evaluated with the same error model.
    The per-rotation error budget of these layouts is OUR adaptation of Litinski (2019) to the GoSC
    geometry (single distance d, two ancilla tiles per multi-patch measurement) - see `gosc_steps`.
"""
import numpy as np
import itertools
import mpmath
from mpmath import mp

mp.prec = 128


def plog_gosc(p, d):
    """GoSC Eq. (10) / Litinski (2019): logical error per patch per code cycle."""
    return mpmath.mpf('0.1') * (100 * mpmath.mpf(p)) ** (mpmath.mpf(d + 1) / 2)

In [ ]:
# ------------------------------------------------------------------ 5-qubit density-matrix engine
class DM5:
    """Density matrix of n (=5) qubits.  backend='ld': numpy extended precision (np.clongdouble, ~19 digits,
    fast; used for the searches); backend='mp': numpy object array of mpmath.mpc (128 bits; used to check
    the transcription against Litinski's published outputs down to 1e-25).  Qubit 1 = most significant bit
    (Litinski's kron ordering)."""

    def __init__(self, n=5, singles=None, backend='ld'):
        self.n, self.backend = n, backend
        N = 2 ** n
        idx = np.arange(N)
        self.bits = [(idx >> (n - 1 - q)) & 1 for q in range(n)]
        if singles is None:
            h = 1 / mpmath.sqrt(2) if backend == 'mp' else np.longdouble(1) / np.sqrt(np.longdouble(2))
            singles = [[h, h]] * n
        if backend == 'mp':
            psi = np.array([mpmath.mpc(1)] * N, dtype=object)
            for q, s in enumerate(singles):
                psi = psi * np.array([mpmath.mpc(s[b]) for b in self.bits[q]], dtype=object)
        else:
            psi = np.ones(N, dtype=np.clongdouble)
            for q, s in enumerate(singles):
                psi = psi * np.array([s[b] for b in self.bits[q]], dtype=np.clongdouble)
        self.rho = np.outer(psi, np.conj(psi))

    def _f(self, x):
        return mpmath.mpf(x) if self.backend == 'mp' else np.longdouble(float(x)) if not isinstance(x, np.longdouble) else x

    def _eig(self, axis):
        s = np.ones(2 ** self.n, dtype=int)
        for q, a in enumerate(axis):
            if a:
                s = s * (1 - 2 * self.bits[q])
        return s

    def rot(self, axis, p1=0, p2=0, p3=0):
        """Z-type Pauli-product rotation exp(i pi/8 P) followed by the faulty-rotation channel of
        Litinski's `apply_rot`: with prob p1 / p2 / p3 the angle is 5pi/8 / -pi/8 / 3pi/8."""
        s = self._eig(axis)
        diff = s[:, None] - s[None, :]            # in {-2, 0, 2}
        if self.backend == 'mp':
            p1, p2, p3 = mpmath.mpf(p1), mpmath.mpf(p2), mpmath.mpf(p3)
            w = [(1 - p1 - p2 - p3, mp.pi / 8), (p1, 5 * mp.pi / 8), (p2, -mp.pi / 8), (p3, 3 * mp.pi / 8)]
            m2 = sum(wk * mpmath.expj(2 * ph) for wk, ph in w)
            M = np.where(diff == 2, m2, np.where(diff == -2, mpmath.conj(m2), mpmath.mpf(1)))
        else:
            p1, p2, p3 = (np.longdouble(float(x)) for x in (p1, p2, p3))
            pi = np.longdouble('3.14159265358979323846264338327950288')
            w = [(1 - p1 - p2 - p3, pi / 8), (p1, 5 * pi / 8), (p2, -pi / 8), (p3, 3 * pi / 8)]
            m2 = sum(wk * (np.cos(2 * ph) + 1j * np.sin(2 * ph)) for wk, ph in w)
            M = np.where(diff == 2, m2, np.where(diff == -2, np.conj(m2), 1)).astype(np.clongdouble)
        self.rho = self.rho * M

    def z_err(self, q, p):
        if p == 0:
            return
        p = mpmath.mpf(p) if self.backend == 'mp' else np.longdouble(float(p))
        sg = (1 - 2 * self.bits[q])
        self.rho = self.rho * np.where(sg[:, None] * sg[None, :] == 1, 1, 1 - 2 * p)

    def x_err(self, q, p):
        if p == 0:
            return
        p = mpmath.mpf(p) if self.backend == 'mp' else np.longdouble(float(p))
        f = np.arange(2 ** self.n) ^ (1 << (self.n - 1 - q))
        self.rho = (1 - p) * self.rho + p * self.rho[np.ix_(f, f)]

    def store(self, px, pz):
        for q in range(self.n):
            self.x_err(q, px[q])
            self.z_err(q, pz[q])

    def postselect_x(self, measured):
        """Project the qubits in `measured` onto |+> (X = +1).  Returns (success probability, normalised
        2x2 density matrix of the remaining qubit)."""
        keep = [q for q in range(self.n) if q not in measured]
        assert len(keep) == 1
        k = keep[0]
        out = np.zeros((2, 2), dtype=self.rho.dtype)
        for a in range(2):
            for b in range(2):
                ia = self.bits[k] == a
                ib = self.bits[k] == b
                out[a, b] = self.rho[np.ix_(ia, ib)].sum() / 2 ** len(measured)
        ps = (out[0, 0] + out[1, 1]).real
        return ps, out / ps


def run_steps(steps, n=5, out_qubit=0, init=None, backend='ld'):
    """Evaluate a protocol given as a list of steps
       ('rot', axis (0/1 tuple), p1, p2, p3)   faulty Z-type pi/8 rotation
       ('store', px (list), pz (list))        X / Z Pauli channels
    on |+>^n; the qubits != out_qubit are measured in X and post-selected on +1.  Returns
    (success probability, output infidelity).  The ideal output is taken from the same circuit without
    errors; the infidelity is evaluated as the population <psi_perp|rho|psi_perp> of the orthogonal
    state (no cancellation in 1 - F)."""
    def go(noisy):
        dm = DM5(n, init, backend)
        for st in steps:
            if st[0] == 'rot':
                dm.rot(st[1], *(st[2:5] if noisy else (0, 0, 0)))
            elif st[0] == 'store' and noisy:
                dm.store(st[1], st[2])
        return dm.postselect_x([q for q in range(n) if q != out_qubit])
    ps0, r0 = go(False)
    # ideal output |psi> from the noiseless (pure) reduced state: rho0 = |psi><psi|
    if backend == 'mp':
        a = mpmath.sqrt(r0[0, 0].real)
        psi = [a, r0[1, 0] / a] if a != 0 else [mpmath.mpc(0), mpmath.mpc(1)]
        perp = [-mpmath.conj(psi[1]), mpmath.conj(psi[0])]
    else:
        a = np.sqrt(r0[0, 0].real)
        psi = [a, r0[1, 0] / a]
        perp = [-np.conj(psi[1]), np.conj(psi[0])]
    ps, r = go(True)
    eps = sum(np.conj(perp[i]) * r[i, j] * perp[j] for i in range(2) for j in range(2)).real
    return ps, eps


def infidelity(rho1, ideal1):
    """1 - <ideal|rho|ideal> for a single-qubit rho (2x2) and an ideal ket."""
    f = sum(np.conj(ideal1[a]) * rho1[a, b] * ideal1[b] for a in range(2) for b in range(2))
    return 1 - f.real

In [ ]:
# ------------------------------------------------------------------ Litinski (2019), transcribed
_Z = 1


def _ax(*qs):
    return tuple(int(q in qs) for q in range(5))


def litinski2019_one_level_steps(p, dx, dz, dm, plog=plog_gosc):
    """Error budget of Litinski's (15-to-1)_{dx,dz,dm} (onelevel15to1.py), qubit 1 = output (index 0)."""
    p = mpmath.mpf(p)
    px, pz, pm = plog(p, dx), plog(p, dz), plog(p, dm)
    t = p / 3
    S = []
    single = lambda q: ('rot', _ax(q), t + 0.5 * (dm / dz) * pz * dm, t + 0.5 * dz * pm, t)
    multi = lambda qs, w: ('rot', _ax(*qs), t + 0.5 * pm * dm, t + 0.5 * pm * dm + 0.5 * w * dx / dm * pm, t)
    sx = lambda c0, c: ('store', [c0] + [c] * 4, [0] * 5)
    S += [single(1), single(2), single(3), multi((1, 2, 3), 3 * dz)]
    S += [('store', [0] + [0.5 * (dz / dx) * px * dm] * 3 + [0], [0] + [0.5 * (dx / dz) * pz * dm] * 3 + [0])]
    S += [multi((0, 1, 2), dx + 2 * dz), multi((0, 1, 3), dx + 3 * dz)]
    S += [('store', [0] * 5, [0.5 * ((dx + 2 * dz) + (dx + 3 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    S += [('store', [0.5 * px * dm] + [0.5 * (dz / dx) * px * dm] * 3 + [0],
           [0.5 * px * dm] + [0.5 * (dx / dz) * pz * dm] * 3 + [0])]
    S += [multi((0, 2, 3), dx + 3 * dz), multi((0, 3, 4), dx + 4 * dz), single(4)]
    S += [('store', [0] * 5, [0.5 * ((dx + 3 * dz) + (dx + 4 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    full = lambda a: ('store', [a * px] + [0.5 * (dz / dx) * px * dm] * 4, [a * px] + [0.5 * (dx / dz) * pz * dm] * 4)
    S += [full(0.5 * dm)]
    S += [multi((0, 1, 4), dx + 4 * dz), multi((0, 2, 4), dx + 4 * dz)]
    S += [('store', [0] * 5, [0.5 * ((dx + 4 * dz) + (dx + 4 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    S += [full(0.5 * dm)]
    S += [multi((0, 1, 2, 3, 4), dx + 4 * dz), multi((2, 3, 4), 3 * dz)]
    S += [('store', [0] * 5, [0.5 * (dx + 4 * dz) / dx * px * dm, 0, 0, 0, 0])]
    S += [full(0.5 * (dm + 2 * dx))]
    S += [multi((1, 3, 4), 4 * dz), multi((1, 2, 4), 4 * dz)]
    S += [('store', [0] + [0.5 * (dz / dx) * px * dm] * 4, [0] + [0.5 * (dx / dz) * pz * dm] * 4)]
    return S


def litinski2019_one_level(p, dx, dz, dm, plog=plog_gosc, backend='ld'):
    ps, eps = run_steps(litinski2019_one_level_steps(p, dx, dz, dm, plog), out_qubit=0, backend=backend)
    qubits = 2 * ((dx + 4 * dz) * 3 * dx + 2 * dm)
    cycles = 6 * dm / ps
    return dict(protocol='Litinski2019 (15-to-1)_{%d,%d,%d}' % (dx, dz, dm), eps=float(eps), p_success=float(ps),
                qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles), levels=1)


def litinski2019_two_level_steps(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog=plog_gosc, backend='ld'):
    p = mpmath.mpf(p)
    px2, pz2, pm2 = plog(p, dx2), plog(p, dz2), plog(p, dm2)
    ps1, pl1 = run_steps(litinski2019_one_level_steps(p, dx, dz, dm, plog), out_qubit=0, backend=backend)
    ps1, pl1 = mpmath.mpf(float(ps1)) if backend != 'mp' else ps1, mpmath.mpf(float(pl1)) if backend != 'mp' else pl1
    pfail = 1 - ps1
    l1time = max(6 * dm / (mpmath.mpf(nl1) / 2) / (1 - pfail), dm2)
    lmove = 10 * dm2 + mpmath.mpf(nl1) / 4 * (dx + 4 * dz)
    R = lambda qs, w: ('rot', _ax(*qs), pl1 + 0.5 * lmove * pm2,
                       0.5 * lmove * pm2 + 0.5 * w * dx2 / dm2 * pm2, 0)
    st = lambda mask, c0=0: ('store', [c0] + [0.5 * (dz2 / dx2) * px2 * l1time if m else 0 for m in mask],
                             [c0] + [0.5 * (dx2 / dz2) * pz2 * l1time if m else 0 for m in mask])
    zs = lambda v: ('store', [0] * 5, [v, 0, 0, 0, 0])
    a = 0.5 * px2 * l1time
    S = [R((1,), dx2 + dz2 + dm2), R((2,), 3 * dz2 + dm2), st([1, 1, 0, 0]),
         R((3,), dx2 + 3 * dz2 + dm2), R((4,), dz2 + dm2), st([1, 1, 1, 1]),
         R((0, 1, 2), dx2 + 2 * dz2 + dm2), R((1, 2, 3), 4 * dz2 + dm2),
         zs(0.5 * (dx2 + 2 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 2, 3), dx2 + 3 * dz2 + dm2), R((0, 1, 3), dx2 + 4 * dz2 + dm2),
         zs(0.5 * ((dx2 + 3 * dz2 + dm2) + (dx2 + 4 * dz2 + dm2)) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 1, 4), dx2 + 4 * dz2 + dm2), R((0, 3, 4), dx2 + 4 * dz2 + dm2),
         zs(0.5 * 2 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 2, 4), dx2 + 4 * dz2 + dm2), R((0, 1, 2, 3, 4), dx2 + 4 * dz2 + dm2),
         zs(0.5 * 2 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((1, 3, 4), dx2 + 4 * dz2 + dm2), R((2, 3, 4), 3 * dz2 + dm2),
         zs(0.5 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], 0.5 * px2 * (dm2 + 2 * dx2)),
         R((1, 2, 4), 4 * dz2 + dm2), st([1, 1, 0, 1])]
    # note: Litinski's code passes the X/Z storage of qubit 1 only where stated (copied as above)
    return S, ps1, pl1, l1time


def litinski2019_two_level(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog=plog_gosc, backend='ld'):
    S, ps1, pl1, l1time = litinski2019_two_level_steps(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog, backend)
    ps2, eps = run_steps(S, out_qubit=0, backend=backend)
    qubits = 2 * int((dx2 + 4 * dz2) * 3 * dx2 + nl1 * ((dx + 4 * dz) * (3 * dx + dm2 / 2) + 2 * dm)
                     + 20 * dm2 * dm2 + 2 * dx2 * dm2)
    cycles = 7.5 * l1time / ps2
    return dict(protocol='Litinski2019 (15-to-1)x(15-to-1)_{%d,%d,%d;%d,%d,%d;%d}' % (dx, dz, dm, dx2, dz2, dm2, nl1),
                eps=float(eps), p_success=float(ps2), qubits=int(qubits), cycles=float(cycles),
                volume=float(qubits * cycles), levels=2, eps_level1=float(pl1))

In [ ]:
# ------------------------------------------------------------------ surface-code p_L calibration (stim)
def noisy_surface_code(d, rounds, basis, p, p_idle=None):
    """Rotated surface-code memory (stim's generated circuit) with the SAME circuit-level noise model and
    time-step convention as the D4 simulation (noisy_d4_lib.NoiseModel / D4SwitchingCircuit rounds):
    X-type ancillas are prepared / measured with RX / MX (no H layers); stim's combined MR is split into a
    measurement step and a reset step, so that one round = reset, 4 CNOT layers, measurement = 6 time
    steps; 2-qubit gates are followed by DEPOLARIZE2(p), resets by a flip, measurements are preceded by a
    flip, and the idle qubits of every time step are depolarized with p_idle."""
    p_idle = p if p_idle is None else p_idle
    base = stim.Circuit.generated('surface_code:rotated_memory_' + basis, distance=d, rounds=rounds).flattened()
    allq, hq = set(), set()
    for ins in base:
        if ins.name in ('R', 'CX', 'M', 'MR', 'H'):
            allq |= {t.value for t in ins.targets_copy()}
        if ins.name == 'H':
            hq |= {t.value for t in ins.targets_copy()}
    out = stim.Circuit()

    def emit(pre, body, post, touched):
        for e in pre:
            out.append(*e)
        for b in body:
            out.append(*b) if isinstance(b, tuple) else out.append(b)
        for e in post:
            out.append(*e)
        idle = sorted(allq - touched)
        if idle and p_idle > 0:
            out.append('DEPOLARIZE1', idle, p_idle)
        out.append('TICK')

    def flush(layer):
        touched, pre, body, post, resets = set(), [], [], [], []
        for ins in layer:
            nm = ins.name
            ts = [t.value for t in ins.targets_copy() if t.is_qubit_target]
            if nm == 'H':
                continue
            if nm in ('R', 'RX'):
                for q in ts:
                    x = q in hq or nm == 'RX'
                    body.append(('RX' if x else 'R', [q]))
                    post.append(('Z_ERROR' if x else 'X_ERROR', [q], p))
            elif nm in ('MR', 'M', 'MX'):
                # one target per instruction keeps stim's measurement-record order
                for q in ts:
                    x = q in hq or nm == 'MX'
                    pre.append(('Z_ERROR' if x else 'X_ERROR', [q], p))
                    body.append(('MX' if x else 'M', [q]))
                    if nm == 'MR':
                        resets.append(q)
            elif nm == 'CX':
                body.append(('CX', ts))
                post.append(('DEPOLARIZE2', ts, p))
            else:
                body.append(ins)
                continue
            touched |= set(ts)
        if not body:
            return                                   # H-only layer: no time step
        emit(pre, body, post, touched)
        if resets:                                   # the reset half of MR: its own time step
            emit([], [('RX' if q in hq else 'R', [q]) for q in resets],
                 [('Z_ERROR' if q in hq else 'X_ERROR', [q], p) for q in resets], set(resets))
    layer = []
    for ins in base:
        if ins.name == 'TICK':
            flush(layer)
            layer = []
        elif ins.name in ('DETECTOR', 'OBSERVABLE_INCLUDE', 'QUBIT_COORDS', 'SHIFT_COORDS'):
            if layer:
                flush(layer)
                layer = []
            out.append(ins)
        else:
            layer.append(ins)
    if layer:
        flush(layer)
    return out


def surface_code_logical_rate(d, p, shots, rounds=None, seed=0):
    """Logical error per round (X + Z memories summed) of the rotated surface code (MWPM)."""
    rounds = rounds or 3 * d
    tot = 0.0
    out = {}
    for basis in ('x', 'z'):
        c = noisy_surface_code(d, rounds, basis, p)
        m = pymatching.Matching.from_detector_error_model(c.detector_error_model(decompose_errors=True))
        det, obs = c.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
        pred = m.decode_batch(det)
        PL = float(np.mean(pred[:, 0] != obs[:, 0]))
        eps = (1 - (1 - 2 * PL) ** (1 / rounds)) / 2 if PL < 0.5 else 0.5
        out[basis] = (PL, eps)
        tot += eps
    return tot, out


def fit_plog(rows):
    """Fit p_L = A (p/p_th)^((d+1)/2) to rows (d, p, p_L) by least squares in log p_L."""
    ds = np.array([r[0] for r in rows], float)
    ps = np.array([r[1] for r in rows], float)
    pl = np.array([r[2] for r in rows], float)
    k = (ds + 1) / 2
    # log pl = log A + k log p - k log pth  -> linear in (log A, log pth)
    A_ = np.stack([np.ones_like(k), -k], axis=1)
    y = np.log(pl) - k * np.log(ps)
    (lA, lpth), *_ = np.linalg.lstsq(A_, y, rcond=None)
    return float(np.exp(lA)), float(np.exp(lpth))


def make_plog(A, pth):
    return lambda p, d: mpmath.mpf(A) * (mpmath.mpf(p) / mpmath.mpf(pth)) ** (mpmath.mpf(d + 1) / 2)

In [ ]:
# ------------------------------------------------------------------ GoSC Sec. 3 layouts with the same model
# the 11 rotations of GoSC Fig. 15 (= msd_lib.fig15_rotations(): columns 5..15 of the 15-to-1 matrix,
# qubit indices 0..4 = qubits 1..5; validated in MSD_15to1_validation M1)
GOSC_ROTATIONS = [(2, 3, 4), (1, 3, 4), (1, 2, 4), (1, 2, 3), (0, 3, 4), (0, 2, 4), (0, 2, 3),
                  (0, 1, 4), (0, 1, 3), (0, 1, 2), (0, 1, 2, 3, 4)]


def gosc_15to1_steps(p, d, plog=plog_gosc, p_in=None, anc_tiles=2, in_channel='pauli'):
    """GoSC Fig. 15 / 17c: qubits 1-4 start in |m> (faulty, error p_in), qubit 5 in |+>; the 11 Z-type
    pi/8 rotations of Fig. 15 (msd_lib.fig15_rotations) are auto-corrected pi/8 rotations of 1 tau = d
    code cycles each; qubits 1-4 are measured in X, qubit 5 is the output.
    Error budget per rotation (our adaptation of Litinski 2019 to the single-distance GoSC layout):
      p1 (P_{pi/2})  = e1 + 0.5 p_L d                        (time-like error of the P x Z_m measurement)
      p2 (P_{-pi/4}) = e2 + 0.5 p_L d + 0.5 anc_tiles p_L d   (space-like errors in the ancilla tiles)
      p3 (P_{pi/4})  = e3
    with (e1, e2, e3) = (p_in/3,)*3 for a raw state with a random Pauli error (in_channel='pauli', GoSC
    Sec. 4 assumption) or (p_in, 0, 0) for a distilled input (in_channel='z', Litinski two-level code);
    every data tile suffers X and Z storage errors 0.5 p_L d per tau."""
    p = mpmath.mpf(p)
    p_in = p if p_in is None else mpmath.mpf(p_in)
    pl = plog(p, d)
    e = (p_in / 3,) * 3 if in_channel == 'pauli' else (p_in, 0, 0)
    S = [('rot', _ax(q), *e) for q in range(4)]               # the four |m> states (0 tau)
    st = ('store', [0.5 * pl * d] * 5, [0.5 * pl * d] * 5)
    for qs in GOSC_ROTATIONS:
        S.append(('rot', _ax(*qs), e[0] + 0.5 * pl * d, e[1] + 0.5 * pl * d + 0.5 * anc_tiles * pl * d, e[2]))
        S.append(st)
    return S


def gosc_15to1(p, d, plog=plog_gosc, backend='ld', **kw):
    """Output error and cost of the GoSC 11-tile 15-to-1 block (Fig. 17c): 11 tiles x 11 tau."""
    ps, eps = run_steps(gosc_15to1_steps(p, d, plog, **kw), out_qubit=4, backend=backend)
    qubits = 11 * 2 * d * d                       # 11 tiles, d^2 data + d^2 measurement qubits each
    cycles = 11 * d / ps
    return dict(protocol='GoSC 15-to-1 (11 tiles x 11 tau), d=%d' % d, eps=float(eps), p_success=float(ps),
                qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles), levels=1, d=d)


def gosc_225to1(p, d1, d2, plog=plog_gosc, move_tiles=5, backend='ld', level1=None):
    """GoSC Fig. 19 (176 tiles, 15 tau): 11 level-1 15-to-1 blocks (distance d1) feed a level-2 15-to-1
    whose 15 rotations (incl. the 4 single-qubit ones) consume distilled states.  Level-1 output error
    enters as a Z-type (P_{pi/2}) error (Litinski two-level code); moving a level-1 state through the
    blue ancilla of Fig. 19 adds 0.5 * move_tiles * p_L(d2) * d2 (our assumption, move_tiles = 5)."""
    ps1, e1 = level1 if level1 is not None else run_steps(gosc_15to1_steps(p, d1, plog), out_qubit=4, backend=backend)
    ps1, e1 = mpmath.mpf(float(ps1)), mpmath.mpf(float(e1))
    pl2 = plog(mpmath.mpf(p), d2)
    mv = 0.5 * move_tiles * pl2 * d2
    S = []
    st = ('store', [0.5 * pl2 * d2] * 5, [0.5 * pl2 * d2] * 5)
    for q in range(4):
        S.append(('rot', _ax(q), e1 + mv + 0.5 * pl2 * d2, 0.5 * pl2 * d2 + pl2 * d2, 0))
        S.append(st)
    for qs in GOSC_ROTATIONS:
        S.append(('rot', _ax(*qs), e1 + mv + 0.5 * pl2 * d2, 0.5 * pl2 * d2 + pl2 * d2, 0))
        S.append(st)
    ps2, eps = run_steps(S, out_qubit=4, backend=backend)
    qubits = 2 * (121 * d1 * d1 + 55 * d2 * d2)   # 11 level-1 blocks of 11 tiles + 55 level-2 tiles
    cycles = 15 * d2 / (ps2 * ps1)                # a level-2 step is skipped when no level-1 state is ready
    return dict(protocol='GoSC 225-to-1 (176 tiles x 15 tau), d1=%d, d2=%d' % (d1, d2), eps=float(eps),
                p_success=float(ps2), qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles),
                levels=2, d=d2, d1=d1, eps_level1=float(e1))

In [ ]:
print('library loaded')

## 1. 設定
* `SCENARIOS`：雑音モデル。A＝アイドル雑音あり（$p_{\rm idle}=p$、SD6 型の標準的な回路雑音）、B＝アイドル雑音なし。
* `KAPPA`：S0, S1, S4 のラウンド数 $\lceil\kappa d\rceil$（[KZH] は $O(d)$ としか書いていない。既定 1、S1 については `KAPPA_SCAN` で感度を見る）。
* `QUIET_POINTS`：**診断用**の追加の点。切り替えの時間窓（D4 相最後のラウンド、$U$ 直前の $Z$ ラウンド、$U$、switch-out、FSC に戻った最初のラウンド）だけ雑音を 0 にする（`quiet_window=True`）。本研究の復号器が 1 個の故障を訂正できない場所はこの窓に集中しており（`validation.ipynb` N5）、この窓の扱いを理想化したときの楽観的な限界を与える。物理的なモデルではない。
* `TIME_CAP`（秒）と `MAX_FAIL`：1 点あたりの打ち切り条件（失敗 200 回で十分な精度）。
* `WORKERS`：並列プロセス数（Colab は 2 程度）。

In [ ]:
QUICK = True
SCENARIOS = {'A': 1.0, 'B': 0.0}            # p_idle = factor * p
P_LIST = [1e-3, 5e-4, 3e-4, 2e-4, 1e-4]
D_LIST = [3, 5, 7, 9]
KAPPA = (1.0, 1.0, 1.0)                     # (kappa0, kappa_in, kappa_out)
KAPPA_SCAN = [(d, 2e-4, 'B', k) for d in (5, 7) for k in (0.4, 0.6, 1.5)]
TIME_CAP, MAX_FAIL, WORKERS = 1200, 200, 4
CALIB_D, CALIB_P = [3, 5, 7, 9, 11], [0.002, 0.003, 0.004, 0.006]
if QUICK:
    P_LIST, D_LIST, KAPPA_SCAN = [1e-3, 3e-4], [3, 5], []
    TIME_CAP, MAX_FAIL, WORKERS = 60, 50, 2
    CALIB_D, CALIB_P = [3, 5, 7], [0.003, 0.006]
SUFFIX = '_quick' if QUICK else ''
MC_CSV, CAL_CSV = f'{RESULTS}/d4_noisy_mc{SUFFIX}.csv', f'{RESULTS}/surface_code_plog{SUFFIX}.csv'
POINTS = [dict(scenario=s, d=d, p=p, p_idle=f * p, kappa=KAPPA) for s, f in SCENARIOS.items() for p in P_LIST for d in D_LIST]
POINTS += [dict(scenario=s, d=d, p=p, p_idle=SCENARIOS[s] * p, kappa=(1.0, k, 1.0)) for d, p, s, k in KAPPA_SCAN]
QUIET_POINTS = [dict(scenario=s, d=d, p=p, p_idle=f * p, kappa=KAPPA, quiet=True, max_fail=100, time_cap=TIME_CAP // 2)
                for s, f in SCENARIOS.items() for p in ([1e-3] if QUICK else [1e-3, 5e-4, 3e-4]) for d in D_LIST]
POINTS += QUIET_POINTS
print(len(POINTS), 'points ->', MC_CSV)

## 2. 回路の規模
各 $d$ について、量子ビット数・時間ステップ数・雑音の位置の数（$p$ をかけると 1 ショットあたりの平均誤り数）。D4 相のラウンドは深い（CCZ を含む 3 つの可換クラス＋Z 検査）ため、同じ $d$ の表面符号より誤りの位置がはるかに多い。

In [ ]:
rows = []
for d in D_LIST:
    R_ = [max(1, int(np.ceil(k * d))) for k in KAPPA]
    circ = D4SwitchingCircuit(d, *R_)
    comp = CompiledProtocol(circ, NoiseModel(1.0, p_idle=1.0))
    blocks = {}
    for b in comp.blocks:
        blocks[b['name'][0]] = blocks.get(b['name'][0], 0) + len(b['moms'])
    rows.append(dict(d=d, qubits=circ.nq, data=circ.nd, time_steps=sum(blocks.values()),
                     **{f'steps_{k}': v for k, v in blocks.items()}, noise_locations=comp.n_loc))
pd.DataFrame(rows)

## 3. モンテカルロ（Kobayashi 方式）
1 ショット：誤りを雑音モデルからサンプルし、Pauli proxy 回路（`FastShot`）を stim の `TableauSimulator` で実行する。各ラウンドの後に flux 復号器（`FluxDecoder`）が補正を決め、物理的に適用する（即時フィードフォワード）。最後に charge 復号器（`ChargeDecoder`）の予測と、雑音なしの最終ラウンド後の $\bar H$ の期待値から不忠実度（0, 1/2, 1 のいずれか）を得る。
結果は塊（chunk）ごとに CSV に追記する（途中で止めても集計できる）。

In [ ]:
import time, multiprocessing as mp_

def run_point(pt, seed):
    d, p = pt['d'], pt['p']
    R_ = [max(1, int(np.ceil(k * d))) for k in pt['kappa']]
    ex = D4Experiment(d, *R_, NoiseModel(p, p_idle=pt['p_idle']), quiet_window=pt.get('quiet', False))
    rng = np.random.default_rng(seed)
    chunk = 400 if d <= 5 else 100 if d == 7 else 40
    out, t_start, nf = [], time.time(), 0
    while time.time() - t_start < pt.get('time_cap', TIME_CAP) and nf < pt.get('max_fail', MAX_FAIL):
        t0 = time.time()
        inf, st = ex.run(chunk, rng)
        nf += int((inf > 0).sum())
        out.append(dict(scenario=pt['scenario'], window='quiet' if pt.get('quiet') else 'noisy', d=d, p=p, p_idle=pt['p_idle'], kappa0=pt['kappa'][0],
                        kappa_in=pt['kappa'][1], kappa_out=pt['kappa'][2], R0=R_[0], Rin=R_[1], Rout=R_[2],
                        shots=chunk, sum_infidelity=float(inf.sum()), n_fail=int((inf > 0).sum()),
                        n_half=int((inf == 0.5).sum()), mean_errors=float(np.mean([s['n_err'] for s in st])),
                        seconds=time.time() - t0))
    return out

def _worker(args):
    return run_point(*args)

t0 = time.time()
tasks = [(pt, SEED + 17 * i) for i, pt in enumerate(POINTS)]
if WORKERS > 1:
    with mp_.get_context('fork').Pool(WORKERS) as pool:
        res = pool.map(_worker, tasks, chunksize=1)
else:
    res = [_worker(t) for t in tasks]
mc = pd.DataFrame([r for rr in res for r in rr])
mc.to_csv(MC_CSV, index=False)
print('wrote', MC_CSV, len(mc), 'chunks, %.0f s' % (time.time() - t0))

## 4. 表面符号の論理誤り率の較正（Litinski 側の $p_L$）
stim が生成する回転表面符号メモリ回路を、D4 と同じ雑音モデル・時間ステップの約束（X 型補助は RX/MX で準備・測定、初期化と測定は別の時間ステップで 1 ラウンド＝初期化＋CNOT 4 層＋測定＝6 時間ステップ、アイドル雑音はシナリオどおり）に置き換え、MWPM で復号する。1 ラウンドあたりの論理誤り（X と Z の和）$p_L$ を求め、`noisy_cost_comparison.ipynb` で $p_L=A\,(p/p_{\rm th})^{(d+1)/2}$ に当てはめる（GoSC 式 (10) と同じ形）。

In [ ]:
rows = []
t0 = time.time()
for scen, f in SCENARIOS.items():
    for d in CALIB_D:
        for p in CALIB_P:
            shots = (400000 if d <= 5 else 200000 if d <= 7 else 100000) // (20 if QUICK else 1)
            tot, o = 0.0, {}
            for basis in ('x', 'z'):
                c = noisy_surface_code(d, 3 * d, basis, p, p_idle=f * p)
                m = pymatching.Matching.from_detector_error_model(c.detector_error_model(decompose_errors=True))
                det, obs = c.compile_detector_sampler(seed=SEED + d * 1000 + int(p * 1e5)).sample(shots, separate_observables=True)
                PL = float(np.mean(m.decode_batch(det)[:, 0] != obs[:, 0]))
                o[basis] = PL
                tot += (1 - (1 - 2 * PL) ** (1 / (3 * d))) / 2
            rows.append(dict(scenario=scen, d=d, p=p, p_idle=f * p, shots=shots, rounds=3 * d,
                             PL_x=o['x'], PL_z=o['z'], pL_per_round=tot))
cal = pd.DataFrame(rows)
cal.to_csv(CAL_CSV, index=False)
print('wrote', CAL_CSV, '%.0f s' % (time.time() - t0))
cal

## 5. 結果の概要
不忠実度の推定値 $\hat\varepsilon_K$ と標準誤差（ショットごとの値 0, 1/2, 1 の平均と標準誤差）。

In [ ]:
g = mc.groupby(['window', 'scenario', 'd', 'p', 'kappa_in']).agg(shots=('shots', 'sum'), s=('sum_infidelity', 'sum'),
                                                       n_fail=('n_fail', 'sum'), n_half=('n_half', 'sum')).reset_index()
g['eps'] = g.s / g.shots
n_one = g.n_fail - g.n_half                      # shots with infidelity 1 (the others have 1/2)
g['stderr'] = np.sqrt(np.maximum((n_one + 0.25 * g.n_half) / g.shots - g.eps ** 2, 0) / g.shots)
g.round(6)